# Notebook 2 — ConfidenceRankedNodes evaluation with bnRep

## Evaluation per generation, complete CPT exports, and strict state auditing

This notebook accepts the **original generation ZIP** produced by `ConfidenceRankedNodes`, without requiring a modified generation notebook. It supports `seismic`, `electricvehicle`, and `gasifier`, identified automatically from the manifest.

**Core integrity rule:** never assume that column `0` of a JSON CPT corresponds to column `0` in bnRep. Each probability is identified **by the node name, state name, and (parent name, parent state) pairs**. Matrices are permuted *only in memory*, after validating their labels. Incompatible labels stop execution rather than produce incorrect metrics.

Outputs include BN KL, CPT KL, MAE, RMSE, JS, Hellinger, and TVD reports, together with `auditoria_estados_bnrep.csv`, `auditoria_alinhamento_geracoes.csv`, and `auditoria_alinhamento_resumo.csv`.

The official bnRep reference is consulted only for **evaluation**. Predicted tables are exported beforehand, directly from the ZIP, without leaking reference CPTs.

**Compatibility:** runtime messages, saved outputs, report text, column names, and input/output conventions remain in Portuguese. Public configuration names and the embedded R exporter are preserved.


In [ ]:
print('AVALIADOR V5 — TRÊS REDES, ALINHAMENTO AUDITADO — MÉTRICAS + CPTS COMPLETAS POR GERAÇÃO / ConfidenceRankedNodes')
print('Aceita schema: qwen35_confidence_ranked_nodes_v1')

AVALIADOR V5 — TRÊS REDES, ALINHAMENTO AUDITADO — MÉTRICAS + CPTS COMPLETAS POR GERAÇÃO / ConfidenceRankedNodes
Aceita schema: qwen35_confidence_ranked_nodes_v1


## 0. Files to upload to Colab

Upload the **original generation ZIP** from Notebook 1, containing `manifesto_geracao.json`, `rede_original.json`, `personas_original.json`, `cpts/`, and `detalhes/`.

Use the original ZIP rather than one previously modified to create CSVs; the **evaluator** handles that export. The official BIF file remains optional for comparison with bnRep. **CPT visualization and CSV export work before installing or consulting the reference.**


In [ ]:
# @title 0.1 — Install Python dependencies
import sys, subprocess, importlib.util, warnings
warnings.filterwarnings('ignore', message=r'.*StructureScore.*deprecated.*', category=FutureWarning)

REQUISITOS = [
    "pgmpy>=1.0,<2",
    "pandas>=2",
    "numpy>=1.26",
    "networkx>=3",
    "scipy>=1.11"
]

print("[COLAB] Instalando/atualizando dependências Python...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", *REQUISITOS]
)

print("[COLAB] Dependências Python prontas.")


[COLAB] Instalando/atualizando dependências Python...
[COLAB] Dependências Python prontas.


In [ ]:
# @title 0.2 — Upload the output from Notebook 1
import os, shutil, zipfile
from pathlib import Path
from google.colab import files

BASE_COLAB = Path("/content")
ENTRADA = BASE_COLAB / "entrada_qwen35_confidence_rn"

if ENTRADA.exists():
    shutil.rmtree(ENTRADA)
ENTRADA.mkdir(parents=True, exist_ok=True)

def _extract_zip_safely(zip_path: Path, destination_path: Path):
    destination_path = destination_path.resolve()
    destination_path.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        for zip_member in zf.infolist():
            target_path = (destination_path / zip_member.filename).resolve()
            if target_path != destination_path and destination_path not in target_path.parents:
                raise RuntimeError(f"ZIP contém caminho inseguro: {zip_member.filename}")
        zf.extractall(destination_path)

print("Envie:")
print("  1) o ZIP da pasta de geração do Notebook 1;")
print("  2) opcionalmente, o BIF oficial da rede.")
uploaded_files = files.upload()

zips = []
bifs = []

for item_name, file_content in uploaded_files.items():
    record_path = ENTRADA / Path(item_name).name
    record_path.write_bytes(file_content)
    if record_path.suffix.lower() == ".zip":
        zips.append(record_path)
    elif record_path.suffix.lower() == ".bif":
        bifs.append(record_path)

if len(zips) != 1:
    raise RuntimeError(
        f"Envie exatamente um ZIP da geração. ZIPs encontrados: {[p.name for p in zips]}"
    )

if len(bifs) > 1:
    raise RuntimeError("Envie no máximo um arquivo BIF.")

_extracted_dir = ENTRADA / "extraido"
_extract_zip_safely(zips[0], _extracted_dir)

provided_bif = bifs[0] if bifs else None

manifest_paths = sorted(_extracted_dir.rglob("manifesto_geracao.json"))
if not manifest_paths:
    raise FileNotFoundError(
        "Não encontrei manifesto_geracao.json. "
        "Envie o ZIP criado ao final do Notebook 1 ConfidenceRankedNodes."
    )

if len(manifest_paths) > 1:
    print("[AVISO] Mais de um manifesto encontrado:")
    for i, m in enumerate(manifest_paths):
        print(f"  [{i}] {m}")
    print("[AVISO] Usando o primeiro. Prefira enviar apenas uma geração por ZIP.")

manifest_path = manifest_paths[0]
generation_directory = manifest_path.parent.resolve()

os.environ["EXPERIMENT_BASE_DIR"] = "/content"
os.environ["PASTA_GERACAO"] = str(generation_directory)
os.environ.setdefault("R_LIBS_USER", "/content/R_libs")
Path(os.environ["R_LIBS_USER"]).mkdir(parents=True, exist_ok=True)

if provided_bif is not None:
    os.environ["BIF_REFERENCIA"] = str(provided_bif.resolve())
    print("[BIF] Referência fornecida:", provided_bif)
else:
    os.environ.pop("BIF_REFERENCIA", None)
    print("[BIF] Nenhum BIF enviado. A referência será recuperada automaticamente pelo bnRep.")

print("\n[OK] Pasta de geração:", generation_directory)
print("[OK] Manifesto:", manifest_path)
for item_name in ("rede_original.json", "personas_original.json", "cpts", "detalhes"):
    p = generation_directory / item_name
    print("  ", "✓" if p.exists() else "✗", p)


Envie:
  1) o ZIP da pasta de geração do Notebook 1;
  2) opcionalmente, o BIF oficial da rede.


Saving resultados_qwen35_9b_electricvehicle_confidence_ranked_nodes.zip to resultados_qwen35_9b_electricvehicle_confidence_ranked_nodes.zip
[BIF] Nenhum BIF enviado. A referência será recuperada automaticamente pelo bnRep.

[OK] Pasta de geração: /content/entrada_qwen35_confidence_rn/extraido
[OK] Manifesto: /content/entrada_qwen35_confidence_rn/extraido/manifesto_geracao.json
   ✓ /content/entrada_qwen35_confidence_rn/extraido/rede_original.json
   ✓ /content/entrada_qwen35_confidence_rn/extraido/personas_original.json
   ✓ /content/entrada_qwen35_confidence_rn/extraido/cpts
   ✓ /content/entrada_qwen35_confidence_rn/extraido/detalhes


In [ ]:
# Second stage: CPU only, without Transformers, a GPU, or text generation.
import os, sys, json, hashlib, itertools, re, shutil, subprocess, time, warnings
warnings.filterwarnings('ignore', message=r'.*StructureScore.*deprecated.*', category=FutureWarning)
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from pgmpy.readwrite import BIFReader
try:
    from pgmpy.models import DiscreteBayesianNetwork as BayesianNetworkModel
except ImportError:
    from pgmpy.models import BayesianNetwork as BayesianNetworkModel

BASE_DIR=Path(os.environ.get(
    'EXPERIMENT_BASE_DIR',
    '/workspace' if Path('/workspace').exists() else str(Path.cwd())
))
manual=os.environ.get('PASTA_GERACAO','').strip()
if not manual:
    raise FileNotFoundError(
        'PASTA_GERACAO não foi configurada. Execute primeiro a célula de upload do ZIP.'
    )

PASTA_GERACAO=Path(manual).resolve()
CAMINHO_MANIFESTO=PASTA_GERACAO/'manifesto_geracao.json'
if not CAMINHO_MANIFESTO.is_file():
    raise FileNotFoundError(CAMINHO_MANIFESTO)

MANIFESTO=json.loads(CAMINHO_MANIFESTO.read_text(encoding='utf-8'))

SCHEMA_ESPERADO='qwen35_confidence_ranked_nodes_v1'
if MANIFESTO.get('schema')!=SCHEMA_ESPERADO:
    raise ValueError(
        f"Manifesto incompatível: recebido {MANIFESTO.get('schema')!r}; "
        f"esperado {SCHEMA_ESPERADO!r}. "
        "Use a saída dos notebooks ConfidenceRankedNodes."
    )

if MANIFESTO.get('configuracao',{}).get('metodo')!='ConfidenceRankedNodes':
    raise ValueError(
        "O manifesto não pertence ao protocolo ConfidenceRankedNodes."
    )
PERMITIR_AVALIACAO_PARCIAL=False
if MANIFESTO['status']!='concluido' and not PERMITIR_AVALIACAO_PARCIAL:
    raise RuntimeError('A geração ainda não terminou. Aguarde todas as condições ou habilite PERMITIR_AVALIACAO_PARCIAL.')
for x in (MANIFESTO['json_rede'],MANIFESTO['json_personas']):
    if not (PASTA_GERACAO/x).is_file():raise FileNotFoundError(PASTA_GERACAO/x)
if hashlib.sha256((PASTA_GERACAO/MANIFESTO['json_rede']).read_bytes()).hexdigest()!=MANIFESTO['configuracao']['json_rede_sha256']:
    raise RuntimeError('O JSON congelado da rede mudou após a geração.')
if hashlib.sha256((PASTA_GERACAO/MANIFESTO['json_personas']).read_bytes()).hexdigest()!=MANIFESTO['configuracao']['json_personas_sha256']:
    raise RuntimeError('O JSON congelado de personas mudou após a geração.')
DIR_AVALIACAO=PASTA_GERACAO/'avaliacao_bnrep'
DIR_AVALIACAO.mkdir(parents=True,exist_ok=True)
NOME_REDE_BNREP=MANIFESTO['nome_rede_bnrep_apenas_identificador']
# Validation metadata; network construction remains dynamic for each network.
REDES_VALIDAS = {
    'seismic': {'nos': 10, 'arestas': 12, 'linhas_cpt': 82},
    'electricvehicle': {'nos': 23, 'arestas': 22, 'linhas_cpt': 79},
    'gasifier': {'nos': 40, 'arestas': 39, 'linhas_cpt': 159},
}
if NOME_REDE_BNREP not in REDES_VALIDAS:
    raise ValueError(
        f'Rede {NOME_REDE_BNREP!r} não prevista nesta versão. '
        f'Redes suportadas: {sorted(REDES_VALIDAS)}'
    )

SEED=MANIFESTO['configuracao']['seed']
RUNPOD_MODE=Path('/workspace').exists()
COLAB_MODE=Path('/content').exists() and not RUNPOD_MODE
BNREP_INSTALACAO='cran'
BNREP_COMMIT='master'
BNREP_REINSTALAR=False
BNREP_FORCAR_EXPORTACAO=False
# Optional: supply the original BIF without installing R in the evaluation environment.
BIF_REFERENCIA=Path(os.environ.get('BIF_REFERENCIA','')) if os.environ.get('BIF_REFERENCIA') else None
print('[GERAÇÃO]',PASTA_GERACAO)
print('[AVALIAÇÃO]',DIR_AVALIACAO)
print('[REDE bnRep]',NOME_REDE_BNREP,'| R só será executado neste notebook, se necessário.')

[GERAÇÃO] /content/entrada_qwen35_confidence_rn/extraido
[AVALIAÇÃO] /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep
[REDE bnRep] electricvehicle | R só será executado neste notebook, se necessário.


In [ ]:
import networkx as nx
from typing import List,Dict,Any,Optional,Tuple

## 1. Parse and validate hybrid CPTs, without an LLM

Evaluation uses the complete CPTs produced by `ConfidenceRankedNodes`.
`bnRep` is accessed only after all predicted CPTs pass the preflight checks.


In [ ]:
# ==========================================================
# NODE INPUT
# ==========================================================
def load_bn_spec(json_path: str) -> Tuple[Dict[str, Dict[str, Any]], List[Tuple[str, str]]]:
    """
    Read a JSON file containing:
    - network: general network metadata, optional
    - nodes: nodes in either the legacy or new format
    - edges: causal relationships already defined by the user

    Accepted legacy format:
    {
      "nodes": {
        "A": {
          "description": "Description of node A.",
          "states": ["False", "True"]
        }
      },
      "edges": [...]
    }

    Accepted new format:
    {
      "network": {...},
      "nodes": [
        {
          "name": "A",
          "definition": "...",
          "entity": "...",
          "scale_type": "ordinal",
          "states": [
            {"name": "Low", "meaning": "..."},
            {"name": "High", "meaning": "..."}
          ],
          "temporal_context": "...",
          "measurement_context": "...",
          "notes": "..."
        }
      ],
      "edges": [...]
    }
    """
    path = Path(json_path)

    # utf-8-sig handles input files that contain a BOM
    with path.open("r", encoding="utf-8-sig") as f:
        data = json.load(f)

    raw_nodes = data.get("nodes")
    raw_edges = data.get("edges", [])
    network_info = data.get("network", {})

    if raw_nodes is None:
        raise ValueError("O JSON precisa conter o campo 'nodes'.")

    nodes: Dict[str, Dict[str, Any]] = {}

    # ======================================================
    # CASE 1: legacy format
    # "nodes": { "A": {"description": "...", "states": [...] } }
    # ======================================================
    if isinstance(raw_nodes, dict):
        for name, value in raw_nodes.items():
            name = str(name).strip()

            if isinstance(value, str):
                nodes[name] = {
                    "description": value.strip(),
                    "states": ["False", "True"],
                }

            elif isinstance(value, dict):
                description = str(value.get("description", name)).strip()
                states = value.get("states", ["False", "True"])

                if not isinstance(states, list) or len(states) < 2:
                    raise ValueError(f"O nó '{name}' precisa ter pelo menos dois estados.")

                nodes[name] = {
                    "description": description,
                    "states": [str(s).strip() for s in states],
                }

            else:
                raise ValueError(f"Formato inválido para o nó '{name}'.")

    # ======================================================
    # CASE 2: new format
    # "nodes": [ {"name": "...", "definition": "...", ... } ]
    # ======================================================
    elif isinstance(raw_nodes, list):
        for node_obj in raw_nodes:
            if not isinstance(node_obj, dict):
                raise ValueError(f"Cada item de 'nodes' precisa ser um objeto. Recebido: {node_obj}")

            name = str(node_obj.get("name", "")).strip()
            if not name:
                raise ValueError(f"Nó sem campo 'name': {node_obj}")

            raw_states = node_obj.get("states", [])
            if not isinstance(raw_states, list) or len(raw_states) < 2:
                raise ValueError(f"O nó '{name}' precisa ter pelo menos dois estados.")

            state_names = []
            state_descriptions = []

            for state in raw_states:
                if isinstance(state, dict):
                    state_name = str(state.get("name", "")).strip()
                    state_meaning = str(state.get("meaning", "")).strip()

                    if not state_name:
                        raise ValueError(f"Estado sem campo 'name' no nó '{name}': {state}")

                    state_names.append(state_name)

                    if state_meaning:
                        state_descriptions.append(f"{state_name}: {state_meaning}")

                else:
                    state_name = str(state).strip()
                    state_names.append(state_name)

            description_parts = []

            network_domain = str(network_info.get("domain", "")).strip()
            network_description = str(network_info.get("description", "")).strip()

            if network_domain:
                description_parts.append(f"Network domain: {network_domain}.")

            if network_description:
                description_parts.append(f"Network description: {network_description}")

            definition = str(node_obj.get("definition", "")).strip()
            entity = str(node_obj.get("entity", "")).strip()
            scale_type = str(node_obj.get("scale_type", "")).strip()
            temporal_context = str(node_obj.get("temporal_context", "")).strip()
            measurement_context = str(node_obj.get("measurement_context", "")).strip()
            notes = str(node_obj.get("notes", "")).strip()

            if definition:
                description_parts.append(f"Definition: {definition}")

            if entity:
                description_parts.append(f"Entity: {entity}")

            if scale_type:
                description_parts.append(f"Scale type: {scale_type}")

            if state_descriptions:
                description_parts.append("State meanings: " + "; ".join(state_descriptions))

            if temporal_context:
                description_parts.append(f"Temporal context: {temporal_context}")

            if measurement_context:
                description_parts.append(f"Measurement context: {measurement_context}")

            if notes:
                description_parts.append(f"Notes: {notes}")

            description = " ".join(description_parts).strip()

            if not description:
                description = name

            nodes[name] = {
                "description": description,
                "states": state_names,
                "raw_metadata": node_obj,
            }

    else:
        raise ValueError("'nodes' deve ser um objeto ou uma lista.")

    edges = parse_and_validate_edges(raw_edges, nodes)

    return nodes, edges

def parse_and_validate_edges(
    raw_edges: List[Any],
    nodes_info: Dict[str, Dict[str, Any]],
) -> List[Tuple[str, str]]:
    """
    Convert and validate the edges supplied in the JSON.

    Accepts:
    - {"source": "A", "target": "B"}
    - ["A", "B"]

    Returns:
    - [("A", "B"), ...]
    """
    known_nodes = set(nodes_info.keys())
    edges: List[Tuple[str, str]] = []

    if not isinstance(raw_edges, list):
        raise ValueError("'edges' deve ser uma lista.")

    for e in raw_edges:
        if isinstance(e, dict):
            src = str(e.get("source", "")).strip()
            dst = str(e.get("target", "")).strip()

        elif isinstance(e, (list, tuple)) and len(e) >= 2:
            src = str(e[0]).strip()
            dst = str(e[1]).strip()

        else:
            raise ValueError(f"Aresta inválida: {e}")

        if src not in known_nodes:
            raise ValueError(f"Aresta inválida: nó de origem desconhecido: {src}")

        if dst not in known_nodes:
            raise ValueError(f"Aresta inválida: nó de destino desconhecido: {dst}")

        if src == dst:
            raise ValueError(f"Aresta inválida: self-loop detectado: {src} -> {dst}")

        edges.append((src, dst))

    # Remove duplicates while preserving order
    deduped = []
    seen = set()

    for edge in edges:
        if edge not in seen:
            seen.add(edge)
            deduped.append(edge)

    # Check that the graph is a DAG
    graph = nx.DiGraph()
    graph.add_nodes_from(nodes_info.keys())
    graph.add_edges_from(deduped)

    if not nx.is_directed_acyclic_graph(graph):
        raise ValueError("As arestas fornecidas geram ciclo. A Rede Bayesiana precisa ser um DAG.")

    return deduped


def node_context_text(
    nodes_info: Dict[str, Dict[str, Any]],
    only_nodes: Optional[List[str]] = None
) -> str:
    selected = only_nodes if only_nodes is not None else list(nodes_info.keys())

    lines = []

    for node in selected:
        info = nodes_info[node]

        states = ", ".join(info["states"])

        lines.append(
            f"- {node}: {info['description']} Possible states: [{states}]"
        )

    return "\n".join(lines)

In [ ]:
# Reconstruct only from the estimated CPT JSONs, without LLM calls.
def load_estimated_cpts(record_path, network_json_path):
    file_path=Path(record_path)
    payload=json.loads(file_path.read_text(encoding='utf-8'))
    if set(payload.get('nodes',{}))!=set(network_json_path['nodes']):
        raise ValueError('Nós divergentes no resultado: '+str(file_path))
    network_edges=[(e['source'],e['target']) for e in payload['edges']]
    if set(network_edges)!=set(network_json_path['edges']):
        raise ValueError('Arestas divergentes no resultado: '+str(file_path))
    bn=BayesianNetworkModel(network_edges)
    bn.add_nodes_from(payload['nodes'])
    if set(payload['cpds'])!=set(payload['nodes']):raise ValueError('CPT de algum nó ausente: '+str(file_path))
    cpds=[]
    for node_name,info in payload['cpds'].items():
        variables=info['variables']
        if variables[0]!=node_name:raise ValueError('CPT com variável principal incompatível: '+node_name)
        parents=variables[1:]
        sn={k:list(v) for k,v in info['state_names'].items()}
        if set(parents)!=set([a for a,b in network_edges if b==node_name]):
            raise ValueError('Pais incompatíveis em '+node_name)
        # Compare state NAMES, never positions or approximate equivalence.
        # This also detects differences in parent-node states.
        for cpd_variable in [node_name] + parents:
            cpd_states = sn.get(cpd_variable)
            network_states = list(network_json_path['nodes'][cpd_variable]['states'])
            if not cpd_states or len(cpd_states) != len(set(cpd_states)):
                raise ValueError(f'CPT de {node_name}: estados inválidos/duplicados em {cpd_variable}')
            if len(cpd_states)!=len(network_states) or set(cpd_states)!=set(network_states):
                raise ValueError(
                    f'CPT de {node_name}: estados de {cpd_variable} incompatíveis com JSON: '
                    f'CPT={cpd_states}, JSON={network_states}'
                )
        vals=np.asarray(info['values'],dtype=float)
        expected_size=int(np.prod([len(sn[p]) for p in parents])) if parents else 1
        if vals.shape!=(len(sn[node_name]),expected_size):
            raise ValueError(f'Formato inválido de {node_name}: {vals.shape}, esperado {(len(sn[node_name]),expected_size)}')
        if np.any(~np.isfinite(vals)) or np.any(vals<0) or not np.allclose(vals.sum(axis=0),1.0,atol=1e-7):
            raise ValueError('CPT com coluna não normalizada: '+node_name)
        args=dict(variable=node_name,variable_card=len(sn[node_name]),values=vals,state_names=sn)
        if parents:args.update(evidence=parents,evidence_card=[len(sn[p]) for p in parents])
        cpds.append(TabularCPD(**args))
    bn.add_cpds(*cpds)
    if not bn.check_model():raise ValueError('Rede JSON não passou no check_model: '+str(file_path))
    return bn


def evaluation_preflight():
    original=json.loads((PASTA_GERACAO/MANIFESTO['json_rede']).read_text(encoding='utf-8-sig'))
    network_nodes,network_edges=load_bn_spec(str(PASTA_GERACAO/MANIFESTO['json_rede']))
    evaluation_outputs=[x for x in MANIFESTO['execucoes'] if x.get('status')=='concluido']
    if not evaluation_outputs:raise RuntimeError('Não há nenhuma condição concluída para avaliar.')
    if not PERMITIR_AVALIACAO_PARCIAL and len(evaluation_outputs)!=len(MANIFESTO['execucoes']):
        raise RuntimeError('Há condições ainda não concluídas; defina PERMITIR_AVALIACAO_PARCIAL=True se necessário.')
    predictions=[]
    for item in evaluation_outputs:
        file_path=PASTA_GERACAO/item['cpt_file']
        if not file_path.is_file():raise FileNotFoundError(file_path)
        if hashlib.sha256(file_path.read_bytes()).hexdigest()!=item['sha256_cpt']:
            raise RuntimeError('CPT alterada após a geração: '+str(file_path))
        network_model=load_estimated_cpts(file_path,{'nodes':network_nodes,'edges':network_edges})
        predictions.append((item,network_model))
    matrix_dimensions = REDES_VALIDAS[NOME_REDE_BNREP]
    if len(network_nodes) != matrix_dimensions['nos'] or len(network_edges) != matrix_dimensions['arestas']:
        raise ValueError(
            f'Estrutura da rede {NOME_REDE_BNREP} inesperada: '
            f'{len(network_nodes)} nós, {len(network_edges)} arestas, '
            f'esperados {matrix_dimensions["nos"]}/{matrix_dimensions["arestas"]}.'
        )
    parents_by_node = {node_name: [] for node_name in network_nodes}
    for parent_name, child_name in network_edges:
        parents_by_node[child_name].append(parent_name)
    expected_cpt_rows = sum(
        int(np.prod([len(network_nodes[p]['states']) for p in parents_by_node[node_name]]))
        for node_name in network_nodes
    )
    if expected_cpt_rows != matrix_dimensions['linhas_cpt']:
        raise ValueError(
            f'Número de linhas de CPT inesperado para {NOME_REDE_BNREP}: '
            f'{expected_cpt_rows} (esperadas {matrix_dimensions["linhas_cpt"]}).'
        )
    print(f'[REDE VALIDADA] {NOME_REDE_BNREP}: {len(network_nodes)} nós, {len(network_edges)} arcos, '
          f'{expected_cpt_rows} linhas CPT por geração.')
    print('[PREFLIGHT] CPTs estimadas válidas:',len(predictions), '— agora podemos consultar bnRep.')
    return network_nodes,network_edges,predictions

## 2. Complete CPTs per generation — **without consulting bnRep**

This section reads **only the final probabilities already generated** (`cpts/*.json`) and their metadata (`detalhes/*.json`). For each node, it enumerates every possible parent configuration, associating matrix values with the distributions and actual state labels.

- **One row** represents one parent configuration of a node. Root nodes have one row.
- **`origem`** identifies `LLM_FULLDIST_ROOT`, `LLM_FULLDIST`, `RANKED_NODES`, or a fallback, as recorded during generation.
- **`confianca`** records the screening confidence when available; it must not be interpreted as demonstrated accuracy.
- **`P(estado)`** contains estimated probabilities in the state order used by the generation JSON.

CSVs are saved in `avaliacao_bnrep/tabelas_cpt_completas/` and automatically included in the **final evaluation ZIP**. The elicitation method **is not run again** here.


In [ ]:
# @title 2.1 — Build complete CPT tables from the ORIGINAL generation ZIP
# No GPU, Qwen calls, or gold/bnRep probabilities.
from itertools import product
from html import escape as escape_html
from IPython.display import HTML, display

DIR_TABELAS_CPT = DIR_AVALIACAO / 'tabelas_cpt_completas'
TABELAS_CPT_GERACOES = {}
TABELA_CPT_COMPLETA = pd.DataFrame()
CAMINHOS_CSV_CPTS = []
PERSONAS_CPT_ATIVAS = []


def _cpt_csv_name(input_value):
    # Use the same naming rule as generation to avoid unsafe names.
    s = re.sub(r'[^a-zA-Z0-9_-]', '_', str(input_value)).strip('_')
    return (s or 'sem_nome')[:110]


def _load_cpt_personas():
    payload = json.loads((PASTA_GERACAO / MANIFESTO['json_personas']).read_text(encoding='utf-8-sig'))
    return [p for p in payload.get('personas', []) if p.get('ativo', True)]


def _validate_cpt_metadata(item, payload):
    metadata_path = PASTA_GERACAO / item.get('metadata_file', '') if item.get('metadata_file') else None
    if metadata_path and metadata_path.is_file():
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    else:
        metadata = payload.get('hybrid_metadata')
    if not isinstance(metadata, dict) or not isinstance(metadata.get('nodes'), dict):
        raise ValueError(f'Não há metadados híbridos válidos para {item["condicao"]} r{item["repeticao"]}')
    return metadata


def _extract_generation_rows(item, persona_role):
    """Build a readable CPT while respecting the CPD parent/state order.

    Validate the numerical matrix, probability sums, and correspondence
    between CPD columns and FullDist/Ranked Nodes metadata rows.
    """
    record_path = PASTA_GERACAO / item['cpt_file']
    if not record_path.is_file():
        raise FileNotFoundError(record_path)
    expected_sha256 = item.get('sha256_cpt')
    if expected_sha256 and hashlib.sha256(record_path.read_bytes()).hexdigest() != expected_sha256:
        raise ValueError(f'O arquivo CPT não corresponde ao SHA256 do manifesto: {record_path}')
    payload = json.loads(record_path.read_text(encoding='utf-8'))
    meta = _validate_cpt_metadata(item, payload)
    cpds = payload.get('cpds', {})
    if not cpds:
        raise ValueError('JSON sem CPTs: ' + str(record_path))
    if set(cpds) != set(payload['nodes']):
        raise ValueError(f'CPTs incompletas em {record_path.name}')
    metanodes = meta['nodes']
    cpt_records = []
    node_counter = 0

    for node_name, cpd in cpds.items():
        node_counter += 1
        var = list(cpd['variables'])
        if not var or var[0] != node_name:
            raise ValueError(f'CPT {node_name}: primeira variável deve ser o nó.')
        parent_names = var[1:]
        states = cpd['state_names']
        ordered_state_names = list(states[node_name])
        # No heuristic spelling/case corrections: differences invalidate the evaluation.
        for variable_name in [node_name]+parent_names:
            predicted_state_labels = list(states[variable_name])
            input_state_labels = list(payload['nodes'][variable_name]['states'])
            if len(predicted_state_labels)!=len(set(predicted_state_labels)) or set(predicted_state_labels)!=set(input_state_labels):
                raise ValueError(f'CPT {node_name}: estados incompatíveis/duplicados em {variable_name}. '
                                 f'CPD={predicted_state_labels}; JSON da geração={input_state_labels}')
        parent_configurations = list(product(*[states[p] for p in parent_names])) if parent_names else [()]
        probability_values = np.asarray(cpd['values'], dtype=float)
        if probability_values.shape != (len(ordered_state_names), len(parent_configurations)):
            raise ValueError(f'CPT {node_name}: dimensões inválidas: {probability_values.shape}')
        if not np.all(np.isfinite(probability_values)) or np.any(probability_values < -1e-12):
            raise ValueError(f'CPT {node_name} contém valores inválidos.')
        if not np.allclose(probability_values.sum(axis=0), 1., atol=1e-7):
            raise ValueError(f'CPT {node_name}: coluna(s) não normalizada(s).')

        if node_name not in metanodes:
            raise ValueError(f'Não há metadados para o nó {node_name}.')
        metadata_rows = metanodes[node_name].get('rows', [])
        row_indices = [int(info['row']) for info in metadata_rows]
        if len(row_indices) != len(parent_configurations) or set(row_indices) != set(range(len(parent_configurations))):
            raise ValueError(f'Metadados incompletos/duplicados em {node_name}: {row_indices}')
        metadata_by_row = {int(info['row']): info for info in metadata_rows}

        for j, config in enumerate(parent_configurations):
            parent_configuration = dict(zip(parent_names, config))
            info = metadata_by_row[j]
            if info.get('parents', {}) != parent_configuration:
                raise ValueError(f'Ordem de pais divergente em {node_name}, linha {j}: {parent_configuration} / {info.get("parents")}')
            metadata_distribution = np.asarray(info.get('distribution', []), dtype=float)
            if metadata_distribution.shape != (len(ordered_state_names),) or not np.allclose(metadata_distribution, probability_values[:, j], atol=1e-7):
                raise ValueError(f'Probabilidades divergentes entre CPD e metadados: {node_name}, linha {j}')

            cpt_record = {
                'modelo': MANIFESTO.get('modelo', ''),
                'rede': NOME_REDE_BNREP,
                'condicao': str(item['condicao']),
                'persona': persona_role,
                'repeticao': int(item['repeticao']),
                'no': node_name,
                'linha_cpt': j + 1,
                'pais': (' | '.join(f'{p}={v}' for p, v in parent_configuration.items())
                         if parent_configuration else 'RAIZ (sem pais)'),
                'n_pais': len(parent_names),
                'configuracao_pais_json': json.dumps(parent_configuration, ensure_ascii=False, sort_keys=True),
                'confianca': info.get('confidence'),
                'origem': info.get('source'),
                'motivo_fallback': info.get('fallback_reason'),
            }
            for e, p in zip(ordered_state_names, probability_values[:, j]):
                cpt_record['P(' + str(e) + ')'] = float(p)
            cpt_records.append(cpt_record)
    return pd.DataFrame(cpt_records)


def export_complete_cpts_from_zip():
    global TABELAS_CPT_GERACOES, TABELA_CPT_COMPLETA, CAMINHOS_CSV_CPTS, PERSONAS_CPT_ATIVAS
    # Use the evaluator preflight checks for matching SHA hashes and a valid network.
    network_nodes, network_edges, predictions = evaluation_preflight()
    expected_size = 0
    parents_by_node = {n: [] for n in network_nodes}
    for parent_name, child_name in network_edges:
        parents_by_node[child_name].append(parent_name)
    for node_name, parent_names in parents_by_node.items():
        row_count = 1
        for parent_name in parent_names:
            row_count *= len(network_nodes[parent_name]['states'])
        expected_size += row_count

    PERSONAS_CPT_ATIVAS = _load_cpt_personas()
    persona_role_names = {p['id']: p.get('papel', p['id']) for p in PERSONAS_CPT_ATIVAS}
    persona_role_names['controle'] = 'Sem persona (controle)'
    DIR_TABELAS_CPT.mkdir(parents=True, exist_ok=True)
    generation_tables = {}
    combined_tables = []
    exported_files = []
    for item, _ in predictions:
        condition_name = str(item['condicao'])
        repetition = int(item['repeticao'])
        entry_key = (condition_name, repetition)
        if entry_key in generation_tables:
            raise ValueError(f'Condição/repetição duplicada no manifesto: {entry_key}')
        df = _extract_generation_rows(item, persona_role_names.get(condition_name, condition_name))
        if len(df) != expected_size:
            raise ValueError(f'CPT incompleta para {entry_key}: {len(df)} linhas; esperadas {expected_size}.')
        csv_path = DIR_TABELAS_CPT / f'cpt_completa_{_cpt_csv_name(condition_name)}_r{repetition}.csv'
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        generation_tables[entry_key] = df
        combined_tables.append(df)
        exported_files.append(csv_path)
        print(f'[CPT] {condition_name} r{repetition}: {len(df)} linhas | {df["no"].nunique()} nós | {csv_path.name}')
    TABELAS_CPT_GERACOES = generation_tables
    TABELA_CPT_COMPLETA = pd.concat(combined_tables, ignore_index=True, sort=False)
    combined_csv_path = DIR_TABELAS_CPT / 'cpt_completa_todas_geracoes.csv'
    TABELA_CPT_COMPLETA.to_csv(combined_csv_path, index=False, encoding='utf-8-sig')
    CAMINHOS_CSV_CPTS = exported_files + [combined_csv_path]
    print(f'[CONSOLIDADO] {len(TABELA_CPT_COMPLETA)} linhas | {len(generation_tables)} gerações | {combined_csv_path.name}')
    return TABELA_CPT_COMPLETA


# Run BEFORE loading the reference. Estimated CPTs only.
TABELA_CPT_COMPLETA = export_complete_cpts_from_zip()

cpt_summary = (
    TABELA_CPT_COMPLETA.groupby(['condicao', 'persona', 'repeticao'], sort=False)
    .agg(nos=('no', 'nunique'), linhas_cpt=('no', 'size'),
         linhas_llm=('origem', lambda s: int(sum(str(x).startswith('LLM_') for x in s))),
         linhas_rn=('origem', lambda s: int(sum(x == 'RANKED_NODES' for x in s))),
         linhas_fallback=('origem', lambda s: int(sum('FALLBACK' in str(x) for x in s))))
    .reset_index()
)
display(cpt_summary)
print('\n[PASTA DOS CSVs]', DIR_TABELAS_CPT)


[REDE VALIDADA] electricvehicle: 23 nós, 22 arcos, 79 linhas CPT por geração.
[PREFLIGHT] CPTs estimadas válidas: 4 — agora podemos consultar bnRep.
[CPT] controle r1: 79 linhas | 23 nós | cpt_completa_controle_r1.csv
[CPT] engenheiro_veiculos_eletricos r1: 79 linhas | 23 nós | cpt_completa_engenheiro_veiculos_eletricos_r1.csv
[CPT] especialista_seguranca_automotiva r1: 79 linhas | 23 nós | cpt_completa_especialista_seguranca_automotiva_r1.csv
[CPT] analista_risco r1: 79 linhas | 23 nós | cpt_completa_analista_risco_r1.csv
[CONSOLIDADO] 316 linhas | 4 gerações | cpt_completa_todas_geracoes.csv


,condicao,persona,repeticao,nos,linhas_cpt,linhas_llm,linhas_rn,linhas_fallback
0,controle,Sem persona (controle),1,23,79,19,60,0
1,engenheiro_veiculos_eletricos,Electric vehicle systems engineer,1,23,79,38,41,0
2,especialista_seguranca_automotiva,Automotive fire safety and reliability specialist,1,23,79,33,46,0
3,analista_risco,Probabilistic risk analyst,1,23,79,32,47,0



[PASTA DOS CSVs] /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep/tabelas_cpt_completas


In [ ]:
# @title 2.2 — Display utilities: per generation and per node
REPETICAO_VISUALIZAR = 1  # Change to 2, 3, etc. if additional repetitions are available.


def _display_cpt_frame(df, display_title):
    """Display ALL rows without truncation, in a scrollable panel."""
    base_columns = ['no', 'linha_cpt', 'pais', 'confianca', 'origem', 'motivo_fallback']
    probability_columns = [c for c in df.columns if c.startswith('P(')]
    display_columns = [c for c in base_columns + probability_columns if c in df.columns]
    display_table = df[display_columns].copy()
    html = display_table.to_html(index=False, escape=True, na_rep='—',
                           float_format=lambda v: f'{v:.6f}', classes='cpt-table')
    display(HTML('''<style>
        .cpt-wrap {max-height:610px;overflow:auto;border:1px solid #cbd5e1;border-radius:7px}
        .cpt-table {border-collapse:collapse;font-family:system-ui,Arial;font-size:12px;min-width:100%;}
        .cpt-table th,.cpt-table td {padding:6px 9px;border-bottom:1px solid #e2e8f0;text-align:left;white-space:nowrap}
        .cpt-table thead th {position:sticky;top:0;background:#edf2f7;color:#202938;z-index:2}
        .cpt-table tr:hover {background:#f1f5f9}
        </style>'''))
    display(HTML('<h4>' + escape_html(display_title) + '</h4>' +
                 '<p>' + str(df['no'].nunique()) + ' CPTs / ' + str(len(df)) +
                 ' linhas. Role o painel para ver a tabela inteira.</p>' +
                 '<div class="cpt-wrap">' + html + '</div>'))


def show_generation(condition_name, repetition=None):
    rep = REPETICAO_VISUALIZAR if repetition is None else int(repetition)
    df = TABELAS_CPT_GERACOES.get((condition_name, rep))
    if df is None:
        print(f'[NÃO ENCONTRADA] {condition_name}, repetição {rep}. Disponíveis:', list(TABELAS_CPT_GERACOES))
        return
    _display_cpt_frame(df, f'{condition_name} — {df["persona"].iat[0]} — repetição {rep}')


def show_node(node_name, condition_name='controle', repetition=None):
    """Display the COMPLETE table for a single node, including every parent configuration."""
    rep = REPETICAO_VISUALIZAR if repetition is None else int(repetition)
    df = TABELAS_CPT_GERACOES.get((condition_name, rep))
    if df is None:
        print('[NÃO ENCONTRADA] Condição/repetição:', condition_name, rep)
        return
    entries = df[df['no'] == node_name]
    if entries.empty:
        print('[NÓ NÃO ENCONTRADO]', node_name, '| disponíveis:', list(df['no']))
        return
    _display_cpt_frame(entries, f'CPT do nó {node_name} — {condition_name} r{rep}')


### 2.3. Control — no persona

This cell displays **all** CPTs from the generation without a persona, including the 159 rows of `gasifier` when that network is evaluated.


In [ ]:
# @title 2.3 — Control (complete table)
show_generation('controle')

no,linha_cpt,pais,confianca,origem,motivo_fallback,P(no),P(yes)
BF,1,RAIZ (sem pais),—,LLM_FULLDIST_ROOT,None,0.500000,0.500000
SCB,1,BF=no | SBB=no | BO=no | CEF=no,0.000000,RANKED_NODES,None,0.937206,0.062794
SCB,2,BF=no | SBB=no | BO=no | CEF=yes,0.000000,RANKED_NODES,None,0.780615,0.219385
SCB,3,BF=no | SBB=no | BO=yes | CEF=no,0.000000,RANKED_NODES,None,0.780615,0.219385
SCB,4,BF=no | SBB=no | BO=yes | CEF=yes,0.000000,RANKED_NODES,None,0.500000,0.500000
SCB,5,BF=no | SBB=yes | BO=no | CEF=no,0.000000,RANKED_NODES,None,0.780615,0.219385
SCB,6,BF=no | SBB=yes | BO=no | CEF=yes,0.000000,RANKED_NODES,None,0.500000,0.500000
SCB,7,BF=no | SBB=yes | BO=yes | CEF=no,0.000000,RANKED_NODES,None,0.500000,0.500000
SCB,8,BF=no | SBB=yes | BO=yes | CEF=yes,0.000000,RANKED_NODES,None,0.219385,0.780615
SCB,9,BF=yes | SBB=no | BO=no | CEF=no,0.000000,RANKED_NODES,None,0.780615,0.219385


### 2.4. Persona 1 — specialty read from the JSON

Displays the complete CPT table for the first active persona.


In [ ]:
# @title 2.4 — Persona 1 (complete table)
if len(PERSONAS_CPT_ATIVAS) >= 1:
    show_generation(PERSONAS_CPT_ATIVAS[0]['id'])
else:
    print('Não há Persona 1 ativa no ZIP.')

no,linha_cpt,pais,confianca,origem,motivo_fallback,P(no),P(yes)
BF,1,RAIZ (sem pais),—,LLM_FULLDIST_ROOT,None,0.990000,0.010000
SCB,1,BF=no | SBB=no | BO=no | CEF=no,0.150000,RANKED_NODES,None,0.937206,0.062794
SCB,2,BF=no | SBB=no | BO=no | CEF=yes,0.150000,RANKED_NODES,None,0.780615,0.219385
SCB,3,BF=no | SBB=no | BO=yes | CEF=no,0.150000,RANKED_NODES,None,0.780615,0.219385
SCB,4,BF=no | SBB=no | BO=yes | CEF=yes,0.150000,RANKED_NODES,None,0.500000,0.500000
SCB,5,BF=no | SBB=yes | BO=no | CEF=no,0.150000,RANKED_NODES,None,0.780615,0.219385
SCB,6,BF=no | SBB=yes | BO=no | CEF=yes,0.150000,RANKED_NODES,None,0.500000,0.500000
SCB,7,BF=no | SBB=yes | BO=yes | CEF=no,0.150000,RANKED_NODES,None,0.500000,0.500000
SCB,8,BF=no | SBB=yes | BO=yes | CEF=yes,0.150000,RANKED_NODES,None,0.219385,0.780615
SCB,9,BF=yes | SBB=no | BO=no | CEF=no,0.150000,RANKED_NODES,None,0.780615,0.219385


### 2.5. Persona 2 — specialty read from the JSON

Displays the complete CPT table for the second active persona.


In [ ]:
# @title 2.5 — Persona 2 (complete table)
if len(PERSONAS_CPT_ATIVAS) >= 2:
    show_generation(PERSONAS_CPT_ATIVAS[1]['id'])
else:
    print('Não há Persona 2 ativa no ZIP.')

no,linha_cpt,pais,confianca,origem,motivo_fallback,P(no),P(yes)
BF,1,RAIZ (sem pais),—,LLM_FULLDIST_ROOT,None,0.990000,0.010000
SCB,1,BF=no | SBB=no | BO=no | CEF=no,0.000000,RANKED_NODES,None,0.937206,0.062794
SCB,2,BF=no | SBB=no | BO=no | CEF=yes,0.000000,RANKED_NODES,None,0.780615,0.219385
SCB,3,BF=no | SBB=no | BO=yes | CEF=no,0.000000,RANKED_NODES,None,0.780615,0.219385
SCB,4,BF=no | SBB=no | BO=yes | CEF=yes,0.000000,RANKED_NODES,None,0.500000,0.500000
SCB,5,BF=no | SBB=yes | BO=no | CEF=no,0.000000,RANKED_NODES,None,0.780615,0.219385
SCB,6,BF=no | SBB=yes | BO=no | CEF=yes,0.000000,RANKED_NODES,None,0.500000,0.500000
SCB,7,BF=no | SBB=yes | BO=yes | CEF=no,0.000000,RANKED_NODES,None,0.500000,0.500000
SCB,8,BF=no | SBB=yes | BO=yes | CEF=yes,0.000000,RANKED_NODES,None,0.219385,0.780615
SCB,9,BF=yes | SBB=no | BO=no | CEF=no,0.000000,RANKED_NODES,None,0.780615,0.219385


### 2.6. Persona 3 — specialty read from the JSON

Displays the complete CPT table for the third active persona.


In [ ]:
# @title 2.6 — Persona 3 (complete table)
if len(PERSONAS_CPT_ATIVAS) >= 3:
    show_generation(PERSONAS_CPT_ATIVAS[2]['id'])
else:
    print('Não há Persona 3 ativa no ZIP.')

no,linha_cpt,pais,confianca,origem,motivo_fallback,P(no),P(yes)
BF,1,RAIZ (sem pais),—,LLM_FULLDIST_ROOT,None,0.999000,0.001000
SCB,1,BF=no | SBB=no | BO=no | CEF=no,0.150000,RANKED_NODES,None,0.937206,0.062794
SCB,2,BF=no | SBB=no | BO=no | CEF=yes,0.150000,RANKED_NODES,None,0.780615,0.219385
SCB,3,BF=no | SBB=no | BO=yes | CEF=no,0.150000,RANKED_NODES,None,0.780615,0.219385
SCB,4,BF=no | SBB=no | BO=yes | CEF=yes,0.150000,RANKED_NODES,None,0.500000,0.500000
SCB,5,BF=no | SBB=yes | BO=no | CEF=no,0.150000,RANKED_NODES,None,0.780615,0.219385
SCB,6,BF=no | SBB=yes | BO=no | CEF=yes,0.150000,RANKED_NODES,None,0.500000,0.500000
SCB,7,BF=no | SBB=yes | BO=yes | CEF=no,0.150000,RANKED_NODES,None,0.500000,0.500000
SCB,8,BF=no | SBB=yes | BO=yes | CEF=yes,0.150000,RANKED_NODES,None,0.219385,0.780615
SCB,9,BF=yes | SBB=no | BO=no | CEF=no,0.150000,RANKED_NODES,None,0.780615,0.219385


### 2.7. Inspect the CPT of a specific node

In addition to the four complete tables above, you can inspect every parent configuration of one node and see which rows came from FullDist or Ranked Nodes. Change the node name to match the network.


In [ ]:
# @title 2.7 — Node CPT (choose a node and condition)
NO_ESCOLHIDO = 'GasifierFailure' if NOME_REDE_BNREP == 'gasifier' else next(iter(TABELA_CPT_COMPLETA['no']))
CONDICAO_ESCOLHIDA = 'controle'
show_node(NO_ESCOLHIDO, CONDICAO_ESCOLHIDA)

no,linha_cpt,pais,confianca,origem,motivo_fallback,P(no),P(yes)
BF,1,RAIZ (sem pais),—,LLM_FULLDIST_ROOT,None,0.500000,0.500000


### 2.8. Check the exported CSV files

Individual CSVs and the consolidated CSV have already been saved. They can also be downloaded in the final evaluation ZIP. This export is **independent of the generation notebook**.


In [ ]:
# @title 2.8 — Exported files and dimensions
for record_path in CAMINHOS_CSV_CPTS:
    print('•', record_path.relative_to(DIR_AVALIACAO), '|', record_path.stat().st_size, 'bytes')
print('\nVisualizar consolidado na memória: TABELA_CPT_COMPLETA')

• tabelas_cpt_completas/cpt_completa_controle_r1.csv | 15455 bytes
• tabelas_cpt_completas/cpt_completa_engenheiro_veiculos_eletricos_r1.csv | 17614 bytes
• tabelas_cpt_completas/cpt_completa_especialista_seguranca_automotiva_r1.csv | 19251 bytes
• tabelas_cpt_completas/cpt_completa_analista_risco_r1.csv | 15982 bytes
• tabelas_cpt_completas/cpt_completa_todas_geracoes.csv | 67894 bytes

Visualizar consolidado na memória: TABELA_CPT_COMPLETA


## 3. Metrics

The existing BN KL and CPT KL conventions are **preserved**, together with MAE, RMSE, and maximum error. Additional metrics do not change the original calculations. Lower values of all distances and divergences are better; Jensen–Shannon and Hellinger are symmetric, whereas KL is not.


In [ ]:
EPS_KL=1e-8

def kl_bits(p,q,eps=EPS_KL):
    p,q=np.asarray(p,dtype=float),np.asarray(q,dtype=float)
    if p.shape!=q.shape or p.ndim!=1 or np.any(p<0) or np.any(q<0):
        raise ValueError('KL: vetores incompatíveis.')
    if p.sum()<=0 or q.sum()<=0:raise ValueError('KL: vetor nulo')
    # Apply the smoothing used in the paper to each distribution and renormalize
    p=(p/p.sum()+eps)/(1+len(p)*eps)
    q=(q/q.sum()+eps)/(1+len(q)*eps)
    return float(np.sum(p*np.log2(p/q)))


def validate_structure_and_states(gold,pred):
    if set(gold.nodes())!=set(pred.nodes()) or set(gold.edges())!=set(pred.edges()):
        raise ValueError('A estrutura/variáveis preditas não coincidem com a gold.')
    for node_name in gold.nodes():
        gp,pp=gold.get_cpds(node_name),pred.get_cpds(node_name)
        if set(gp.variables)!=set(pp.variables):raise ValueError(f'Pais incompatíveis: {node_name}')
        for v in gp.variables:
            if gp.state_names[v]!=pp.state_names[v]:
                raise ValueError(f'Ordem dos estados de {v} não coincide: {gp.state_names[v]} != {pp.state_names[v]}')


def iter_cpt_rows(gold,node_name):
    parents=list(gold.get_parents(node_name))
    domains=[gold.get_cpds(p).state_names[p] for p in parents]
    for c in itertools.product(*domains):
        yield parents,dict(zip(parents,c))


def _cpt_vector(cpd,node_name,config):
    return np.array([float(cpd.get_value(**{node_name:s,**config}))
                     for s in cpd.state_names[node_name]],dtype=float)


def kl_metrics(gold,pred):
    validate_structure_and_states(gold,pred)
    inf=VariableElimination(gold)
    weighted_sum=0.0; local_metrics=[]
    for node_name in gold.nodes():
        gc,pc=gold.get_cpds(node_name),pred.get_cpds(node_name)
        parents=list(gold.get_parents(node_name))
        marg=inf.query(variables=parents,joint=True,show_progress=False) if parents else None
        for _,config in iter_cpt_rows(gold,node_name):
            kl=kl_bits(_cpt_vector(gc,node_name,config),_cpt_vector(pc,node_name,config))
            w=float(marg.get_value(**config)) if parents else 1.0
            weighted_sum+=w*kl
            local_metrics.append({'node':node_name,'parents':config,'KL_bits':kl,'peso_marginal':w})
    return {'BN KL Divergence':float(weighted_sum),
            'CPT KL Divergence':float(np.mean([z['KL_bits'] for z in local_metrics])),
            'detalhe':pd.DataFrame(local_metrics)}


def uniform_cpds(network_model):
    new_network=BayesianNetworkModel(network_model.edges());new_network.add_nodes_from(network_model.nodes())
    cpds=[]
    for node_name in network_model.nodes():
        c=network_model.get_cpds(node_name)
        numcols=int(np.prod(c.cardinality[1:])) if len(c.cardinality)>1 else 1
        cpds.append(TabularCPD(variable=node_name,variable_card=c.variable_card,
                               values=np.ones((c.variable_card,numcols))/c.variable_card,
                               evidence=list(c.variables[1:]) or None,
                               evidence_card=list(map(int,c.cardinality[1:])) or None,
                               state_names={v:list(c.state_names[v]) for v in c.variables}))
    new_network.add_cpds(*cpds);assert new_network.check_model();return new_network


def random_cpds(network_model,seed=SEED):
    rng=np.random.default_rng(seed)
    new_network=BayesianNetworkModel(network_model.edges());new_network.add_nodes_from(network_model.nodes())
    cpds=[]
    for node_name in network_model.nodes():
        c=network_model.get_cpds(node_name)
        numcols=int(np.prod(c.cardinality[1:])) if len(c.cardinality)>1 else 1
        vals=rng.random((c.variable_card,numcols));vals/=vals.sum(axis=0)
        cpds.append(TabularCPD(variable=node_name,variable_card=c.variable_card,
                               values=vals,evidence=list(c.variables[1:]) or None,
                               evidence_card=list(map(int,c.cardinality[1:])) or None,
                               state_names={v:list(c.state_names[v]) for v in c.variables}))
    new_network.add_cpds(*cpds);assert new_network.check_model();return new_network

## 4. Official bnRep reference — **only at this stage**


In [ ]:
RESULTADOS=DIR_AVALIACAO
PASTA_BNREP=DIR_AVALIACAO/'referencia_bnrep'
PASTA_BNREP.mkdir(parents=True,exist_ok=True)

In [ ]:
# ==============================================================
# R / bnRep -> BIF (actual ground truth; no manually entered probabilities)
# ==============================================================
import shutil, sys, textwrap, subprocess, os, hashlib
from pathlib import Path

PASTA_BNREP = RESULTADOS / 'bnrep_referencias'
PASTA_BNREP.mkdir(exist_ok=True, parents=True)

R_EXPORTADOR = r"""
args <- commandArgs(trailingOnly = TRUE)
if (length(args) != 7L) stop('Esperados 7 argumentos: nome, bif, rds, info, modo, commit, reinstalar')
nome <- args[[1]]
bif <- args[[2]]
rds <- args[[3]]
info <- args[[4]]
modo <- args[[5]]
commit <- args[[6]]
reinstalar <- (args[[7]] == 'true')
repos <- 'https://cloud.r-project.org'
options(repos = c(CRAN = repos))
# Biblioteca R local do ambiente de avaliação (Colab ou outro ambiente).
r_libs <- Sys.getenv('R_LIBS_USER', unset='')
if (nzchar(r_libs)) { dir.create(r_libs, recursive=TRUE,showWarnings=FALSE); .libPaths(c(r_libs,.libPaths())) }

# bnRep importa Rgraphviz (Bioconductor). No Colab, garantimos essa dependência primeiro.
if (!requireNamespace('Rgraphviz', quietly = TRUE)) {
  if (!requireNamespace('BiocManager', quietly = TRUE))
    utils::install.packages('BiocManager', repos = repos)
  BiocManager::install('Rgraphviz', ask = FALSE, update = FALSE)
}

if (!requireNamespace('bnlearn', quietly = TRUE)) {
  utils::install.packages('bnlearn', repos = repos)
}

if (reinstalar || !requireNamespace('bnRep', quietly = TRUE)) {
  if (modo == 'github') {
    if (!requireNamespace('pak', quietly = TRUE))
      utils::install.packages('pak', repos = repos)
    if (!requireNamespace('pak', quietly = TRUE)) stop('Não foi possível instalar pak.')
    pacote <- sprintf('manueleleonelli/bnRep@%s', commit)
    pak::pak(pacote, ask = FALSE)
  } else if (modo == 'cran') {
    utils::install.packages('bnRep', repos = repos, dependencies = NA)
  } else {
    stop(paste('Modo desconhecido:', modo))
  }
}
if (!requireNamespace('bnRep', quietly = TRUE)) stop('bnRep indisponível.')
if (!requireNamespace('bnlearn', quietly = TRUE)) stop('bnlearn indisponível.')

# data() carrega os parâmetros originais, sem refazer estimação no R.
env <- new.env(parent = baseenv())
utils::data(list = nome, package = 'bnRep', envir = env)
if (!exists(nome, envir = env, inherits = FALSE)) {
  stop(sprintf('Rede %s não encontrada em bnRep.', nome))
}
referencia <- get(nome, envir = env, inherits = FALSE)
if (!inherits(referencia, 'bn.fit')) stop('A referência não é um bn.fit.')
if (any(vapply(referencia, function(no) is.null(no$prob), logical(1))))
  stop('A rede tem nó sem CPT discreta. Escolha uma rede totalmente discreta.')

# Assinatura comprovada na documentação bnlearn: write.bif(file, fitted).
bnlearn::write.bif(bif, referencia)
saveRDS(referencia, rds)

# Auditoria simples, sem exigir pacotes adicionais.
remotesha <- utils::packageDescription('bnRep')$RemoteSha
if (is.null(remotesha)) remotesha <- 'nao_disponivel_instalacao_cran'
graph <- bnlearn::bn.net(referencia)
info_linhas <- c(
  sprintf('rede=%s', nome),
  sprintf('bnRep_version=%s', as.character(utils::packageVersion('bnRep'))),
  sprintf('bnRep_RemoteSha=%s', remotesha),
  sprintf('bnlearn_version=%s', as.character(utils::packageVersion('bnlearn'))),
  sprintf('R_version=%s', as.character(getRversion())),
  sprintf('nos=%d', length(names(referencia))),
  sprintf('arestas=%d', nrow(bnlearn::arcs(graph))),
  sprintf('classe=%s', paste(class(referencia), collapse = ',')),
  sprintf('bif=%s', bif)
)
writeLines(info_linhas, info, useBytes = TRUE)
cat(paste(info_linhas, collapse = '\n'), '\n')
"""


def _ensure_rscript():
    rscript = shutil.which('Rscript')
    if rscript:
        return rscript
    if not COLAB_MODE and not RUNPOD_MODE:
        raise RuntimeError('Rscript não encontrado. Instale r-base ou execute preparar_runpod.sh.')
    if not hasattr(os, 'geteuid') or os.geteuid() != 0:
        raise RuntimeError('Rscript ausente sem root. Execute preparar_runpod.sh como root no terminal ou instale R.')
    print('[R] Não encontrado. Instalando r-base no ambiente (requer root); aguarde...',flush=True)
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run([
        'apt-get', 'install', '-y', '-qq',
        'r-base', 'r-base-dev',
        'libcurl4-openssl-dev', 'libssl-dev', 'libxml2-dev',
        'graphviz', 'libgraphviz-dev', 'libgit2-dev'
    ], check=True)
    rscript = shutil.which('Rscript')
    if not rscript:
        raise RuntimeError('Instalação do R terminou, mas Rscript não foi encontrado.')
    return rscript


def export_bnrep_reference(item_name=NOME_REDE_BNREP, directory_path=PASTA_BNREP,
                              installation_mode=BNREP_INSTALACAO, commit=BNREP_COMMIT,
                              force_export=BNREP_FORCAR_EXPORTACAO):
    directory_path = Path(directory_path)
    directory_path.mkdir(parents=True, exist_ok=True)
    bif = directory_path / f'{item_name}.bif'
    rds = directory_path / f'{item_name}.rds'
    info = directory_path / f'{item_name}_versoes_R.txt'
    script = directory_path / 'exportar_referencia_bnrep.R'
    script.write_text(R_EXPORTADOR, encoding='utf-8')
    if bif.exists() and bif.stat().st_size > 0 and not force_export and not BNREP_REINSTALAR:
        print('[bnRep] BIF existente reutilizado:', bif)
    else:
        r_command = [_ensure_rscript(), str(script), str(item_name), str(bif), str(rds),
                   str(info), str(installation_mode), str(commit), str(BNREP_REINSTALAR).lower()]
        print('[bnRep] Exportando rede oficial por R: ', item_name)
        print('Na primeira execução a instalação pode levar muitos minutos.')
        print('[bnRep] Instalando/carregando pacote e exportando referência. Acompanhe a saída:', flush=True)
        log_r = directory_path / f'{item_name}_instalacao_R.log'
        with log_r.open('w', encoding='utf-8') as log:
            with subprocess.Popen(r_command, text=True, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, bufsize=1) as proc:
                for line_index, line in enumerate(proc.stdout, start=1):
                    log.write(line)
                    log.flush()
                    if line_index <= 12 or line_index % 25 == 0 or any(
                            output_keyword in line.lower() for output_keyword in ('error', 'warning', 'erro', 'instal', 'rede=', 'version=')):
                        print('[R]', line.rstrip(), flush=True)
                return_code = proc.wait(timeout=7200)
        if return_code:
            raise RuntimeError(f'R falhou (código {return_code}). Log: {log_r}')
        print('[bnRep] Referência exportada. Log completo:', log_r, flush=True)
    # Proceed only if the BIF exists and produces a valid network.
    ref = BIFReader(str(bif)).get_model()
    if not ref.check_model():
        raise RuntimeError(f'O BIF {bif} não passou na validação pgmpy.')
    manifest = {'nome': item_name, 'origem': 'bnRep',
                'modo_instalacao': installation_mode, 'commit_solicitado': commit,
                'arquivo_bif': str(bif), 'sha256_bif': hashlib.sha256(bif.read_bytes()).hexdigest(),
                'nodes': len(ref.nodes()), 'edges': len(ref.edges()),
                'arquivo_rds': str(rds) if rds.exists() else None,
                'informacoes_R': info.read_text(encoding='utf-8') if info.exists() else None}
    (directory_path/f'{item_name}_manifesto.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    print('[OK] Referência verificada: ', manifest['nodes'], 'nós, ',manifest['edges'],'arestas.')
    return bif, ref


## 5. State validation and alignment (required)

bnRep may store states in a different order from the generation JSON:

- **`seismic`**: typically qualitative states such as `Low / Medium / High` and `Short / Medium / Long`. Each label must match literally.
- **`electricvehicle`**: generation JSON uses `no / yes`; the bnRep reference often uses `yes / no`.
- **`gasifier`**: the JSON uses `NotOccurred / Occurred`; the bnRep reference may use a different order.

Evaluation does not assume that the first position has the same meaning. A CPD with parents is compared using combinations of **child state names and each parent's state names**, reordering the matrix *in memory* before computing KL, MAE, RMSE, and the other metrics. Original generation tables and official CPTs **are not overwritten**.

There are two levels of verification: exact matching of label sets and an independent numerical audit of every probability before and after permutation. Reports are saved in `avaliacao_bnrep/auditoria_alinhamento/`. Any mismatch stops metric computation.


In [ ]:
def long_cpt_table(ref):
    entries=[]
    for node_name in ref.nodes():
        cpd=ref.get_cpds(node_name)
        for _,config in iter_cpt_rows(ref,node_name):
            for state_label in cpd.state_names[node_name]:
                entries.append({'no':node_name,'configuracao_pais':json.dumps(config,ensure_ascii=False,sort_keys=True),
                               'estado':state_label,'probabilidade_referencia':float(cpd.get_value(**{node_name:state_label,**config}))})
    return pd.DataFrame(entries)


def compare_cpts_side_by_side(ref,pred):
    validate_structure_and_states(ref,pred)
    entries=[]
    for node_name in ref.nodes():
        gc,pc=ref.get_cpds(node_name),pred.get_cpds(node_name)
        for _,config in iter_cpt_rows(ref,node_name):
            for state_label in gc.state_names[node_name]:
                entry_key={node_name:state_label,**config}
                reference_probability=float(gc.get_value(**entry_key))
                estimated_probability=float(pc.get_value(**entry_key))
                entries.append({'no':node_name,'configuracao_pais':json.dumps(config,ensure_ascii=False,sort_keys=True),
                               'estado':state_label,'probabilidade_referencia':reference_probability,
                               'probabilidade_llama':estimated_probability,
                               'erro_assinado':estimated_probability-reference_probability,'erro_absoluto':abs(estimated_probability-reference_probability)})
    return pd.DataFrame(entries)

In [ ]:
def _new_model_with_cpds(reference_model, cpds):
    m = BayesianNetworkModel(list(reference_model.edges()))
    m.add_nodes_from(reference_model.nodes())
    m.add_cpds(*cpds)
    if not m.check_model():
        raise RuntimeError('Alinhamento gerou CPT inválida.')
    return m


def _validate_labels(state_labels, validation_context):
    state_labels = list(state_labels)
    if not state_labels or len(state_labels) != len(set(state_labels)):
        raise ValueError(f'Rótulos vazios/duplicados em {validation_context}: {state_labels}')
    if not all(isinstance(s, str) and s.strip() and s == s.strip() for s in state_labels):
        raise ValueError(f'Rótulo inválido em {validation_context}: {state_labels}')
    return state_labels


def validate_reference_label_sets(gold, nodes_info, edges):
    """Ensure the same network and LITERAL labels; report permutations without applying them."""
    if set(gold.nodes()) != set(nodes_info):
        raise ValueError('NÓS NÃO COINCIDEM. Só gold=' + str(sorted(set(gold.nodes())-set(nodes_info)))
                         + '; só JSON=' + str(sorted(set(nodes_info)-set(gold.nodes()))))
    if set(gold.edges()) != set(edges):
        raise ValueError('ARESTAS NÃO COINCIDEM. Só gold=' + str(sorted(set(gold.edges())-set(edges)))
                         + '; só JSON=' + str(sorted(set(edges)-set(gold.edges()))))
    state_permutations = []
    for node_name in gold.nodes():
        reference_state_labels = _validate_labels(gold.get_cpds(node_name).state_names[node_name], 'bnRep/' + node_name)
        payload = _validate_labels(nodes_info[node_name]['states'], 'JSON/' + node_name)
        if len(reference_state_labels) != len(payload) or set(reference_state_labels) != set(payload):
            raise ValueError(f'ESTADOS DIFERENTES em {node_name}: JSON={payload}; bnRep={reference_state_labels}')
        if reference_state_labels != payload:
            state_permutations.append({'no': node_name, 'ordem_json': payload, 'ordem_gold': reference_state_labels,
                            'indices_json_na_ordem_gold': [payload.index(s) for s in reference_state_labels]})
    return state_permutations


def align_prediction_in_memory(pred, gold):
    """Reorder CHILD states and PARENT CONFIGURATIONS by label.

    Address every cell using pred_cpd.get_value(node="State", parent1="State", ...),
    never by its position in the input array. Original files remain unchanged.
    """
    if set(pred.nodes()) != set(gold.nodes()) or set(pred.edges()) != set(gold.edges()):
        raise ValueError('Arestas/nós da estimativa diferem da rede bnRep selecionada.')
    cpds = []
    for node_name in gold.nodes():
        gold_cpd, pred_cpd = gold.get_cpds(node_name), pred.get_cpds(node_name)
        gold_parents = list(gold_cpd.variables[1:])
        predicted_parents = list(pred_cpd.variables[1:])
        if set(gold_parents) != set(predicted_parents):
            raise ValueError(f'Conjunto de pais diferente no nó {node_name}')
        for v in [node_name] + gold_parents:
            g = _validate_labels(gold_cpd.state_names[v], f'gold/{node_name}/{v}')
            p = _validate_labels(pred_cpd.state_names[v], f'pred/{node_name}/{v}')
            if len(g) != len(p) or set(g) != set(p):
                raise ValueError(f'Rótulos incompatíveis em {node_name}/{v}: gold={g}; pred={p}')
        gold_domains = [list(gold_cpd.state_names[p]) for p in gold_parents]
        configs = itertools.product(*gold_domains) if gold_parents else [()]
        probability_matrix = np.asarray([
            [float(pred_cpd.get_value(**{node_name: state_label, **dict(zip(gold_parents, config))}))
             for state_label in gold_cpd.state_names[node_name]]
            for config in configs
        ], dtype=float).T
        kwargs = dict(variable=node_name, variable_card=len(gold_cpd.state_names[node_name]),
                      values=probability_matrix,
                      state_names={v: list(gold_cpd.state_names[v]) for v in [node_name]+gold_parents})
        if gold_parents:
            kwargs.update(evidence=gold_parents,
                          evidence_card=[len(gold_cpd.state_names[p]) for p in gold_parents])
        cpds.append(TabularCPD(**kwargs))
    aligned_network = _new_model_with_cpds(gold, cpds)
    validate_structure_and_states(gold, aligned_network)
    return aligned_network


def audit_alignment_by_label(original_prediction, gold, aligned_prediction, condition_name, repetition):
    """Check EVERY probability by name before/after alignment and return audit rows.

    This check independently verifies the alignment result and covers
    permutations of the parent order as well as the child state order.
    """
    validate_structure_and_states(gold, aligned_prediction)
    alignment_audit = []
    for node_name in gold.nodes():
        reference_cpd = gold.get_cpds(node_name)
        original_cpd = original_prediction.get_cpds(node_name)
        aligned_cpd = aligned_prediction.get_cpds(node_name)
        parent_names = list(reference_cpd.variables[1:])
        state_domains = [list(reference_cpd.state_names[p]) for p in parent_names]
        configs = itertools.product(*state_domains) if parent_names else [()]
        probability_count = 0
        max_alignment_delta = 0.0
        for config in configs:
            conf_dict = dict(zip(parent_names, config))
            for state_label in reference_cpd.state_names[node_name]:
                state_assignment = {node_name: state_label, **conf_dict}
                before_value = float(original_cpd.get_value(**state_assignment))
                after_value = float(aligned_cpd.get_value(**state_assignment))
                if not (np.isfinite(before_value) and np.isfinite(after_value)):
                    raise ValueError(f'Probabilidade não finita: {node_name}/{state_assignment}')
                delta = abs(before_value - after_value)
                max_alignment_delta = max(max_alignment_delta, delta)
                probability_count += 1
                if delta > 1e-10:
                    raise AssertionError(
                        f'ERRO DE ALINHAMENTO: {node_name}, {state_assignment}, '
                        f'original={before_value}, alinhado={after_value}, diferença={delta}'
                    )
        original_child_states = list(original_cpd.state_names[node_name])
        gold_child_states = list(reference_cpd.state_names[node_name])
        child_states_changed = original_child_states != gold_child_states
        parent_order_changed = list(original_cpd.variables[1:]) != parent_names
        parent_states_changed = any(
            list(original_cpd.state_names[p]) != list(reference_cpd.state_names[p]) for p in parent_names
        )
        alignment_audit.append({
            'condicao': condition_name, 'repeticao': int(repetition), 'no': node_name,
            'ordem_estados_predicao': json.dumps(original_child_states, ensure_ascii=False),
            'ordem_estados_bnrep': json.dumps(gold_child_states, ensure_ascii=False),
            'permutacao_indices_filho': json.dumps([original_child_states.index(s) for s in gold_child_states]),
            'ordem_pais_predicao': json.dumps(list(original_cpd.variables[1:]), ensure_ascii=False),
            'ordem_pais_bnrep': json.dumps(parent_names, ensure_ascii=False),
            'algum_estado_filho_invertido': bool(child_states_changed),
            'algum_estado_pai_invertido': bool(parent_states_changed),
            'ordem_pais_alterada': bool(parent_order_changed),
            'n_probabilidades_verificadas': probability_count,
            'erro_maximo_pos_alinhamento': max_alignment_delta,
            'resultado': 'OK — equivalência por rótulo verificada',
        })
    return alignment_audit


## 6. Run evaluation and organize results by generation

Each individual generation produces a subfolder such as `por_geracao/controle/r1/` or `por_geracao/persona_01_<id>/r1/`. Ensembles are kept separate from individual results. In addition to per-generation reports, this stage saves consolidated comparisons and condition-level aggregates, which are useful when `repeticoes > 1`.


In [ ]:
import math

def _save_json_atomic(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    temp.replace(path)


def _error_statistics(probability_errors):
    if probability_errors is None or len(probability_errors)==0:
        return {}
    e=probability_errors['erro_assinado'].to_numpy(dtype=float)
    ae=np.abs(e)
    return {
        'MAE':float(np.mean(ae)),
        'RMSE':float(np.sqrt(np.mean(e**2))),
        'erro_maximo':float(np.max(ae)),
        'MSE':float(np.mean(e**2)),
        'vies_medio':float(np.mean(e)),
        'erro_absoluto_mediana':float(np.median(ae)),
        'erro_absoluto_p90':float(np.percentile(ae,90)),
        'pct_probabilidades_erro_ate_005':float(100*np.mean(ae<=0.05)),
        'pct_probabilidades_erro_ate_010':float(100*np.mean(ae<=0.10)),
        'n_probabilidades':int(len(ae)),
    }


def _distribution_distances(p,q):
    p=np.asarray(p,dtype=float)
    q=np.asarray(q,dtype=float)
    if p.shape!=q.shape or np.any(p<0) or np.any(q<0):
        raise ValueError('Distribuições incompatíveis para métricas adicionais.')
    p=p/p.sum();q=q/q.sum()
    midpoint_distribution=(p+q)/2
    # Jensen–Shannon in bits, using the same smoothing policy as KL.
    js=0.5*kl_bits(p,midpoint_distribution)+0.5*kl_bits(q,midpoint_distribution)
    return {
        'TVD':float(0.5*np.sum(np.abs(p-q))),
        'Hellinger':float(np.sqrt(0.5*np.sum((np.sqrt(p)-np.sqrt(q))**2))),
        'JS_bits':float(max(0.0,js)),
    }


def summarize_metrics(gold,pred):
    # BN_KL and CPT_KL retain exactly the same definitions as before.
    metric_values=kl_metrics(gold,pred)
    probability_errors=compare_cpts_side_by_side(gold,pred)
    probability_errors=probability_errors.copy()
    probability_errors['probabilidade_estimada']=probability_errors['probabilidade_llama']
    probability_errors['erro_quadratico']=probability_errors['erro_assinado']**2
    dist_rows=[]
    for idx,row in metric_values['detalhe'].iterrows():
        node_name=row['node'];config=row['parents']
        p=_cpt_vector(gold.get_cpds(node_name),node_name,config)
        q=_cpt_vector(pred.get_cpds(node_name),node_name,config)
        dist_rows.append(_distribution_distances(p,q))
    for metric_field in ('TVD','Hellinger','JS_bits'):
        metric_values['detalhe'][metric_field]=[x[metric_field] for x in dist_rows]
    basic_metrics=_error_statistics(probability_errors)
    local_metrics=metric_values['detalhe']
    extras={
        'CPT_KL_mediana_bits':float(local_metrics['KL_bits'].median()),
        'CPT_KL_p90_bits':float(local_metrics['KL_bits'].quantile(.9)),
        'TVD_media':float(local_metrics['TVD'].mean()),
        'TVD_mediana':float(local_metrics['TVD'].median()),
        'Hellinger_media':float(local_metrics['Hellinger'].mean()),
        'JS_media_bits':float(local_metrics['JS_bits'].mean()),
        'JS_p90_bits':float(local_metrics['JS_bits'].quantile(.9)),
        'n_linhas_cpt':int(len(local_metrics)),
    }
    return metric_values,probability_errors,{
        'BN_KL_bits':float(metric_values['BN KL Divergence']),
        'CPT_KL_media_bits':float(metric_values['CPT KL Divergence']),
        **basic_metrics,**extras,
    }



def _item_hybrid_metadata(item):
    rel=item.get('metadata_file')
    if not rel:
        return None
    p=PASTA_GERACAO/rel
    if not p.is_file():
        return None
    return json.loads(p.read_text(encoding='utf-8'))


def _source_confidence_map(meta):
    lookup_map={}
    if not meta:
        return lookup_map
    for node_name,nm in meta.get('nodes',{}).items():
        for row in nm.get('rows',[]):
            parent_names=row.get('parents') or {}
            entry_key=(node_name,json.dumps(parent_names,ensure_ascii=False,sort_keys=True))
            lookup_map[entry_key]={
                'origem_linha':row.get('source'),
                'confianca_linha':row.get('confidence'),
            }
    return lookup_map


def _hybrid_summary(item,meta):
    llm_row_count=int(item.get('linhas_llm',0) or 0)
    rn_row_count=int(item.get('linhas_ranked_nodes',0) or 0)
    total=llm_row_count+rn_row_count

    conf=[]
    source_counts={}
    if meta:
        for nm in meta.get('nodes',{}).values():
            for row in nm.get('rows',[]):
                src=row.get('source')
                source_counts[src]=source_counts.get(src,0)+1
                c=row.get('confidence')
                if c is not None:
                    try:
                        conf.append(float(c))
                    except Exception:
                        pass

    fallback=int(sum(v for k,v in source_counts.items() if k and 'FALLBACK' in str(k).upper() and 'UNIFORM' in str(k).upper()))
    return {
        'linhas_fallback_uniforme':fallback,
        'percentual_fallback_uniforme':100.0*fallback/max(1,len([r for n in (meta or {}).get('nodes',{}).values() for r in n.get('rows',[])])),
        'linhas_llm':llm_row_count,
        'linhas_ranked_nodes':rn_row_count,
        'linhas_total_hibrido':total,
        'percentual_llm':(100.0*llm_row_count/total) if total else np.nan,
        'percentual_ranked_nodes':(100.0*rn_row_count/total) if total else np.nan,
        'confianca_media':float(np.mean(conf)) if conf else np.nan,
        'confianca_mediana':float(np.median(conf)) if conf else np.nan,
        'confianca_min':float(np.min(conf)) if conf else np.nan,
        'confianca_max':float(np.max(conf)) if conf else np.nan,
        'n_confiancas':len(conf),
        'fontes_linhas':source_counts,
    }



def _condition_information():
    file_path=PASTA_GERACAO/MANIFESTO['json_personas']
    payload=json.loads(file_path.read_text(encoding='utf-8-sig'))
    active_persona_list=[p for p in payload.get('personas',[]) if p.get('ativo',True)]
    lookup_map={}
    for i,p in enumerate(active_persona_list,start=1):
        item_name=str(p['id'])
        persona_role=str(p.get('papel') or item_name)
        lookup_map[item_name]={
            'grupo_condicao':'Persona',
            'persona_numero':i,
            'persona_id':item_name,
            'persona_papel':persona_role,
            'persona_especialidade':p.get('especialidade',''),
            'rotulo_condicao':f'Persona {i} — {persona_role}',
        }
    return payload,lookup_map


def _condition_label(condition_name,lookup_map):
    if condition_name=='controle':
        return {'grupo_condicao':'Controle','persona_numero':0,'persona_id':'',
                'persona_papel':'Sem persona','persona_especialidade':'',
                'rotulo_condicao':'Controle — sem persona'}
    if condition_name in lookup_map:
        return dict(lookup_map[condition_name])
    if condition_name=='ensemble_iguais':
        condition_text='Ensemble — pesos iguais'
    elif condition_name=='ensemble_ponderados':
        condition_text='Ensemble — pesos das personas'
    else:
        condition_text=f'Condição — {condition_name}'
    return {'grupo_condicao':'Ensemble' if condition_name.startswith('ensemble_') else 'Outro',
            'persona_numero':None,'persona_id':'',
            'persona_papel':'Fusão sem consultas adicionais' if condition_name.startswith('ensemble_') else '',
            'persona_especialidade':'','rotulo_condicao':condition_text}


def _safe_name(input_value):
    item_name=re.sub(r'[^a-zA-Z0-9_-]+','_',str(input_value)).strip('_')
    return item_name[:110] or 'sem_nome'


def _condition_directory(condition_name,rep,lookup_map):
    info=_condition_label(condition_name,lookup_map)
    if info['grupo_condicao']=='Controle':
        directory_path='controle'
    elif info['grupo_condicao']=='Persona':
        directory_path=f"persona_{info['persona_numero']:02d}_{_safe_name(condition_name)}"
    else:
        directory_path=_safe_name(condition_name)
    return DIR_AVALIACAO/'por_geracao'/directory_path/f'r{rep}'


def _save_csv(path,entries):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(entries).to_csv(path,index=False,encoding='utf-8-sig')


def _save_text(path,text_content):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(text_content,encoding='utf-8')


def _format_report_value(x):
    if x is None:
        return '—'
    try:
        if pd.isna(x):return '—'
    except (TypeError,ValueError):
        pass
    return f'{x:.6f}' if isinstance(x,(int,float,np.number)) else str(x)


def _individual_report(meta,main_metrics,hybrid_metrics,node_metrics,source_metrics):
    entries=[f"# {meta['rotulo_condicao']}",
            f"**Modelo:** {meta['modelo']} | **Rede:** {meta['rede']} | **Repetição:** {meta['repeticao']}",
            '', '## Métricas gerais desta geração',
            '| Métrica | Valor |','|---|---:|']
    for k,v in main_metrics.items():
        entries.append(f'| {k} | {_format_report_value(v)} |')
    entries+=['','## Resumo híbrido','| Indicador | Valor |','|---|---:|']
    for k,v in hybrid_metrics.items():
        if isinstance(v,dict):continue
        entries.append(f'| {k} | {_format_report_value(v)} |')
    entries+=['','## Métricas por origem',
             'Consultar `metricas_por_origem.csv` para LLM FullDist, raízes, Ranked Nodes e fallbacks.',
             '', '## Métricas por nó',
             'Consultar `metricas_por_no.csv`, `metricas_por_linha.csv` e `erros_por_probabilidade.csv`.',
             '', '> Os grupos por origem contêm linhas diferentes; não interpretar seus erros como comparação causal pareada.',
             '> Fallback uniforme é um prior técnico e não uma estimativa do Qwen.']
    return '\\n'.join(entries)+'\\n'


def _aggregate_repetitions(df):
    if df.empty:return pd.DataFrame()
    metric_values=['BN_KL_bits','CPT_KL_media_bits','MAE','RMSE','erro_maximo',
              'MSE','vies_medio','erro_absoluto_mediana','erro_absoluto_p90',
              'CPT_KL_mediana_bits','CPT_KL_p90_bits','TVD_media',
              'Hellinger_media','JS_media_bits','JS_p90_bits',
              'pct_probabilidades_erro_ate_005','pct_probabilidades_erro_ate_010']
    group_keys=['modelo','rede','metodo','condicao','rotulo_condicao','grupo_condicao',
            'persona_numero','persona_id','persona_papel']
    entries=[]
    for keys,metric_group in df.groupby(group_keys,dropna=False,sort=False):
        reg=dict(zip(group_keys,keys));reg['n_geracoes']=len(metric_group)
        for metric_name in metric_values:
            if metric_name not in metric_group:continue
            probability_values=pd.to_numeric(metric_group[metric_name],errors='coerce').dropna()
            if probability_values.empty:continue
            reg[f'{metric_name}_media']=float(probability_values.mean())
            reg[f'{metric_name}_desvio_padrao']=float(probability_values.std(ddof=1)) if len(probability_values)>1 else np.nan
            reg[f'{metric_name}_mediana']=float(probability_values.median())
            reg[f'{metric_name}_min']=float(probability_values.min())
            reg[f'{metric_name}_max']=float(probability_values.max())
        entries.append(reg)
    return pd.DataFrame(entries)


def _rank_metrics(df):
    if df.empty:return pd.DataFrame()
    metric_values=['BN_KL_bits','CPT_KL_media_bits','MAE','RMSE','erro_maximo',
              'TVD_media','Hellinger_media','JS_media_bits','CPT_KL_p90_bits']
    entries=[]
    for (model_id,network_model,method_name,rep),metric_group in df.groupby(['modelo','rede','metodo','repeticao'],dropna=False):
        for m in metric_values:
            if m not in metric_group:continue
            ranked_rows=metric_group.sort_values(m,kind='stable')
            for pos,(_,row) in enumerate(ranked_rows.iterrows(),start=1):
                entries.append({'modelo':model_id,'rede':network_model,'metodo':method_name,
                               'repeticao':rep,'metrica':m,'menor_melhor':True,
                               'posicao':pos,'condicao':row['condicao'],
                               'rotulo_condicao':row['rotulo_condicao'],
                               'grupo_condicao':row['grupo_condicao'],
                               'valor':float(row[m])})
    return pd.DataFrame(entries)


def _overall_report(df,lookup_map):
    entries=[f'# Relatório de avaliação — {NOME_REDE_BNREP}',
            f'Modelo: **{MANIFESTO.get("modelo")}** | Método: **ConfidenceRankedNodes**',
            '', '## Identificação das condições',
            '| Condição | Nome/papel | Tipo |','|---|---|---|',
            '| controle | Sem persona | Controle |']
    for pid,info in lookup_map.items():
        entries.append(f"| {pid} | {info['rotulo_condicao']} | Persona {info['persona_numero']} |")
    entries +=['| ensemble_iguais | Fusão com pesos iguais | Ensemble |',
             '| ensemble_ponderados | Fusão com pesos declarados | Ensemble |',
             '', '## Comparação de métricas principais',
             '| Geração | BN KL (bits) ↓ | CPT KL (bits) ↓ | MAE ↓ | RMSE ↓ | JS média (bits) ↓ | Hellinger ↓ |',
             '|---|---:|---:|---:|---:|---:|---:|']
    for _,r in df.iterrows():
        report_fields=[r['rotulo_condicao']]+[_format_report_value(r.get(k)) for k in
            ['BN_KL_bits','CPT_KL_media_bits','MAE','RMSE','JS_media_bits','Hellinger_media']]
        entries.append('| '+' | '.join(report_fields)+' |')
    entries+=['','## Organização dos resultados',
             '- `comparacao_resumo.csv`: formato anterior mantido, com mais colunas.',
             '- `comparacao_geral.csv`: quadro geral por geração, incluindo nome e especialidade da persona.',
             '- `comparacao_geracoes_individuais.csv`: controle + cada persona; sem ensembles.',
             '- `comparacao_ensembles.csv`: apenas fusões, sem chamadas adicionais ao LLM.',
             '- `estatisticas_por_condicao.csv`: médias e desvio padrão se houver repetições.',
             '- `ranking_por_metrica.csv`: classificação descritiva separada por métrica e repetição.',
             '- `indice_geracoes.csv`: caminho da pasta de cada geração.',
             '- `por_geracao/`: tabelas e relatório Markdown de cada controle/persona/ensemble.',
             '', '## Cuidados de interpretação',
             'KL global usa marginais da rede referência. Erros por origem são observacionais: as linhas LLM/RN são selecionadas de maneira distinta, não pareadas.',
             'Confiança é autorrelatada, não calibrada. Priors de fallback devem ser identificados como tal.',
             'Uma única repetição não permite inferir estabilidade ou significância estatística.',
             'A referência bnRep é consultada somente depois da geração e do preflight.',
            ]
    return '\\n'.join(entries)+'\\n'


def _metric_dictionary():
    return '''# Dicionário de métricas

Menores valores indicam menor erro/distância, exceto percentuais de acerto dentro da tolerância (maiores são melhores).

| Campo | Definição |
|---|---|
| BN_KL_bits | Soma das divergências KL por linha ponderadas pela probabilidade das configurações parentais na rede bnRep (métrica original). |
| CPT_KL_media_bits | Média aritmética de KL(gold || estimada) das linhas de CPT (métrica original). |
| CPT_KL_mediana_bits / CPT_KL_p90_bits | Mediana e percentil 90 de KL por linha. |
| MAE | Média de |p_estimada - p_bnRep| sobre todas as probabilidades. |
| RMSE | Raiz da média de (p_estimada - p_bnRep)^2. |
| MSE | Média do erro ao quadrado; RMSE^2. |
| erro_maximo | Maior erro absoluto em uma entrada de CPT. |
| vies_medio | Média do erro com sinal (estimada - real) nas entradas das CPTs. |
| erro_absoluto_mediana / erro_absoluto_p90 | Mediana e percentil 90 do erro absoluto por probabilidade. |
| pct_probabilidades_erro_ate_005 / _010 | Percentual das probabilidades cujo erro absoluto é até 0,05 ou 0,10 (em pontos da escala [0,1]). |
| TVD_media / TVD_mediana | Média/mediana da distância de variação total por linha: 0,5 × soma dos erros absolutos na distribuição. Intervalo [0,1]. |
| Hellinger_media | Média por linha de sqrt(0,5 × soma((sqrt(p)−sqrt(q))²)). Intervalo [0,1]. |
| JS_media_bits / JS_p90_bits | Jensen–Shannon (bits) por linha: 0,5 KL(p||m) + 0,5 KL(q||m); m=(p+q)/2. Tem suavização numérica consistente com a KL já adotada. |
| confianca_media | Confiança autorrelatada nas linhas em que foi registrada, não é probabilidade calibrada de acerto. |
| linhas_fallback_uniforme | Número de linhas cuja origem/metadados indicam prior uniforme de contingência; não é estimativa do LLM. |

A saída `metricas_por_origem_llm_rn.csv` compara grupos **distintos** de linhas, já separados pela triagem. Não permite atribuir causalmente superioridade ao LLM ou ao Ranked Nodes.\n'''

def evaluate_and_save():
    network_nodes,network_edges,predictions=evaluation_preflight()  # No LLM is imported in this notebook.

    if BIF_REFERENCIA:
        bif=Path(BIF_REFERENCIA)
        if not bif.is_file():
            raise FileNotFoundError(bif)
        gold=BIFReader(str(bif)).get_model()
        if not gold.check_model():
            raise RuntimeError('BIF fornecido inválido.')
        reference_source='bif_fornecido'
    else:
        bif,gold=export_bnrep_reference(
            item_name=NOME_REDE_BNREP,
            directory_path=DIR_AVALIACAO/'referencia_bnrep'
        )
        reference_source='bnRep_R'

    state_permutations=validate_reference_label_sets(gold,network_nodes,network_edges)
    alignment_audit_dir = DIR_AVALIACAO/'auditoria_alinhamento'
    alignment_audit_dir.mkdir(parents=True, exist_ok=True)
    state_audit_records=[]
    for node_name in gold.nodes():
        json_states=list(network_nodes[node_name]['states'])
        gold_states=list(gold.get_cpds(node_name).state_names[node_name])
        state_audit_records.append({
            'no':node_name,
            'ordem_json':json.dumps(json_states, ensure_ascii=False),
            'ordem_bnrep':json.dumps(gold_states, ensure_ascii=False),
            'indices_json_na_ordem_bnrep':json.dumps([json_states.index(s) for s in gold_states]),
            'permutacao_necessaria':json_states!=gold_states,
            'estados_compativeis':True,
        })
    pd.DataFrame(state_audit_records).to_csv(
        alignment_audit_dir/'auditoria_estados_bnrep.csv',index=False,encoding='utf-8-sig'
    )
    print(f'[ALINHAMENTO] {len(state_permutations)} de {len(state_audit_records)} nós '
          'têm ordem de estados diferente entre o JSON e o bnRep.')
    if state_permutations:
        print('[ALINHAMENTO] Amostra de permutações:',state_permutations[:3])
    all_alignment_audits=[]

    long_cpt_table(gold).to_csv(
        DIR_AVALIACAO/'CPTs_originais_bnRep.csv',
        index=False,
        encoding='utf-8-sig'
    )

    evaluation_outputs=[]
    all_row_metrics=[]
    all_node_metrics=[]
    all_probability_errors=[]
    all_source_metrics=[]
    all_hybrid_metrics=[]
    generation_index=[]
    persona_config,persona_map=_condition_information()
    _save_csv(DIR_AVALIACAO/'mapa_personas.csv',[
        {'condicao':pid,**info,'peso':next((p.get('peso') for p in persona_config['personas'] if p['id']==pid),None)}
        for pid,info in persona_map.items()
    ])

    def record_result(item,pred,derived=False,persona_weights=None):
        original_prediction=pred
        pred=align_prediction_in_memory(original_prediction,gold)
        # Stop BEFORE computing MAE/KL if any probability changes when addressed by label.
        generation_audits=audit_alignment_by_label(
            original_prediction,gold,pred,item['condicao'],item['repeticao']
        )
        all_alignment_audits.extend(generation_audits)
        m,probability_errors,main_metrics=summarize_metrics(gold,pred)

        cond=item['condicao']
        method_name=item['metodo']
        rep=item['repeticao']
        item_name=f'{cond}_{method_name}_r{rep}'

        meta=None if derived else _item_hybrid_metadata(item)
        lookup_map=_source_confidence_map(meta)

        # Associate each probability and CPT row with its LLM/RN source.
        if not derived and lookup_map:
            probability_errors=probability_errors.copy()
            probability_errors['origem_linha']=probability_errors.apply(
                lambda r: lookup_map.get(
                    (r['no'],r['configuracao_pais']),{}
                ).get('origem_linha'),
                axis=1
            )
            probability_errors['confianca_linha']=probability_errors.apply(
                lambda r: lookup_map.get(
                    (r['no'],r['configuracao_pais']),{}
                ).get('confianca_linha'),
                axis=1
            )
        else:
            probability_errors=probability_errors.copy()
            probability_errors['origem_linha']='FUSAO' if derived else None
            probability_errors['confianca_linha']=np.nan

        node_metrics_frame=m['detalhe'].copy()
        node_metrics_frame['configuracao_pais']=node_metrics_frame['parents'].apply(
            lambda x: json.dumps(x,ensure_ascii=False,sort_keys=True)
        )

        if not derived and lookup_map:
            node_metrics_frame['origem_linha']=node_metrics_frame.apply(
                lambda r: lookup_map.get(
                    (r['node'],r['configuracao_pais']),{}
                ).get('origem_linha'),
                axis=1
            )
            node_metrics_frame['confianca_linha']=node_metrics_frame.apply(
                lambda r: lookup_map.get(
                    (r['node'],r['configuracao_pais']),{}
                ).get('confianca_linha'),
                axis=1
            )
        else:
            node_metrics_frame['origem_linha']='FUSAO' if derived else None
            node_metrics_frame['confianca_linha']=np.nan

        # KL and MAE per node.
        node_metrics=node_metrics_frame.groupby('node',as_index=False).agg(
            n_linhas_cpt=('KL_bits','size'),
            CPT_KL_media_bits=('KL_bits','mean'),
            CPT_KL_mediana_bits=('KL_bits','median'),
            CPT_KL_max_bits=('KL_bits','max'),
            TVD_media=('TVD','mean'),
            Hellinger_media=('Hellinger','mean'),
            JS_media_bits=('JS_bits','mean'),
        ).rename(columns={'node':'no'})
        node_error_metrics=probability_errors.groupby('no',as_index=False).agg(
            MAE=('erro_absoluto','mean'),
            MSE=('erro_quadratico','mean'),
            erro_maximo=('erro_absoluto','max'),
            vies_medio=('erro_assinado','mean'),
            n_probabilidades=('erro_assinado','size'),
        )
        node_error_metrics['RMSE']=np.sqrt(node_error_metrics['MSE'])
        node_metrics=node_metrics.merge(node_error_metrics,on='no',how='left')

        entries=[]
        for _,row in node_metrics_frame.iterrows():
            entries.append({
                'no':row['node'],
                'pais':row['parents'],
                'KL_bits':float(row['KL_bits']),
                'TVD':float(row['TVD']),
                'Hellinger':float(row['Hellinger']),
                'JS_bits':float(row['JS_bits']),
                'peso_marginal_gold':float(row['peso_marginal']),
                'origem_linha':row.get('origem_linha'),
                'confianca_linha':(
                    None if pd.isna(row.get('confianca_linha'))
                    else float(row.get('confianca_linha'))
                ),
            })

        hybrid_metrics=_hybrid_summary(item,meta) if not derived else {
            'linhas_fallback_uniforme':np.nan,
            'percentual_fallback_uniforme':np.nan,
            'linhas_llm':np.nan,
            'linhas_ranked_nodes':np.nan,
            'linhas_total_hibrido':np.nan,
            'percentual_llm':np.nan,
            'percentual_ranked_nodes':np.nan,
            'confianca_media':np.nan,
            'confianca_mediana':np.nan,
            'confianca_min':np.nan,
            'confianca_max':np.nan,
            'n_confiancas':0,
            'fontes_linhas':{},
        }

        condition_info=_condition_label(cond,persona_map)
        condition_metadata={
            'condicao':cond,
            **condition_info,
            'metodo':method_name,
            'repeticao':rep,
            'modelo':MANIFESTO['modelo'],
            'rede':NOME_REDE_BNREP,
            'tipo':'fusao_sem_novas_consultas' if derived else 'geracao_individual',
            'pesos_aplicados':persona_weights,
            'sem_dados_observados':True,
            'bnrep_apenas_avaliacao':True,
            'modelo_nao_carregado_nesta_etapa':True,
            'source_cpt_json':item.get('cpt_file'),
            'source_metadata_json':item.get('metadata_file'),
            'source_console_log':item.get('console_log'),
            'familiaridade_arquivo':MANIFESTO.get('familiaridade_arquivo'),
        }

        # Metrics by source (LLM versus RN).
        source_metrics=[]
        if not derived and lookup_map:
            source_groups=sorted(
                x for x in probability_errors['origem_linha'].dropna().unique().tolist()
            )
            for src in source_groups:
                ep=probability_errors[probability_errors['origem_linha']==src]
                lp=node_metrics_frame[node_metrics_frame['origem_linha']==src]
                reg={
                    'condicao':cond,
                    **condition_info,
                    'metodo':method_name,
                    'repeticao':rep,
                    'modelo':MANIFESTO['modelo'],
                    'origem_linha':src,
                    'n_probabilidades':int(len(ep)),
                    'n_linhas_cpt':int(len(lp)),
                    'MAE':float(ep['erro_absoluto'].mean()) if len(ep) else np.nan,
                    'RMSE':float(np.sqrt(np.mean(ep['erro_assinado'].to_numpy(dtype=float)**2))) if len(ep) else np.nan,
                    'erro_maximo':float(ep['erro_absoluto'].max()) if len(ep) else np.nan,
                    'CPT_KL_media_bits':float(lp['KL_bits'].mean()) if len(lp) else np.nan,
                    'TVD_media':float(lp['TVD'].mean()) if len(lp) else np.nan,
                    'Hellinger_media':float(lp['Hellinger'].mean()) if len(lp) else np.nan,
                    'JS_media_bits':float(lp['JS_bits'].mean()) if len(lp) else np.nan,
                    'MSE':float(np.mean(ep['erro_assinado'].to_numpy(dtype=float)**2)) if len(ep) else np.nan,
                    'vies_medio':float(ep['erro_assinado'].mean()) if len(ep) else np.nan,
                    'confianca_media':float(lp['confianca_linha'].dropna().mean())
                        if lp['confianca_linha'].notna().any() else np.nan,
                }
                source_metrics.append(reg)
                all_source_metrics.append(reg)

        probability_error_records=probability_errors.to_dict(orient='records')
        cpt_record={
            **condition_metadata,
            'metricas':main_metrics,
            'resumo_hibrido':hybrid_metrics,
            'metricas_por_origem':source_metrics,
            'metricas_por_no':node_metrics.to_dict(orient='records'),
            'metricas_por_linha':entries,
            'erros_por_probabilidade':probability_error_records,
        }

        dest=DIR_AVALIACAO/'relatorios'/f'metricas_{item_name}.json'
        _save_json_atomic(dest,cpt_record)

        evaluation_outputs.append({
            **condition_metadata,
            **main_metrics,
            **{k:v for k,v in hybrid_metrics.items() if k!='fontes_linhas'},
            'relatorio_json':str(dest.relative_to(DIR_AVALIACAO)),
        })

        if not derived:
            all_hybrid_metrics.append({
                'condicao':cond,
                **condition_info,
                'metodo':method_name,
                'repeticao':rep,
                'modelo':MANIFESTO['modelo'],
                **hybrid_metrics,
            })

        for x in entries:
            all_row_metrics.append({
                'condicao':cond,
                **condition_info,
                'metodo':method_name,
                'repeticao':rep,
                **x
            })

        for x in node_metrics.to_dict(orient='records'):
            all_node_metrics.append({
                'condicao':cond,
                **condition_info,
                'metodo':method_name,
                'repeticao':rep,
                **x
            })

        for x in probability_error_records:
            all_probability_errors.append({
                'condicao':cond,
                **condition_info,
                'metodo':method_name,
                'repeticao':rep,
                **x
            })

        condition_folder=_condition_directory(cond,rep,persona_map)
        condition_folder.mkdir(parents=True,exist_ok=True)
        _save_csv(condition_folder/'resumo_metricas.csv',[{**condition_metadata,**main_metrics,
                     **{k:v for k,v in hybrid_metrics.items() if k!='fontes_linhas'}}])
        _save_csv(condition_folder/'metricas_por_no.csv',node_metrics.to_dict(orient='records'))
        _save_csv(condition_folder/'metricas_por_linha.csv',entries)
        _save_csv(condition_folder/'erros_por_probabilidade.csv',probability_error_records)
        _save_csv(condition_folder/'metricas_por_origem.csv',source_metrics)
        _save_csv(condition_folder/'resumo_hibrido.csv',[hybrid_metrics])
        _save_text(condition_folder/'LEIA_ME.md',
            _individual_report(condition_metadata,main_metrics,hybrid_metrics,node_metrics,source_metrics))
        generation_index.append({
            'condicao':cond,**condition_info,'metodo':method_name,'repeticao':rep,
            'modelo':MANIFESTO.get('modelo'),'rede':NOME_REDE_BNREP,
            'diretorio':str(condition_folder.relative_to(DIR_AVALIACAO)),
            'relatorio_individual':str((condition_folder/'LEIA_ME.md').relative_to(DIR_AVALIACAO)),
        })

        print(
            f"[MÉTRICAS] {cond}/{method_name}/r{rep}: "
            f"BN_KL={main_metrics['BN_KL_bits']:.6f} "
            f"CPT_KL={main_metrics['CPT_KL_media_bits']:.6f} "
            f"MAE={main_metrics['MAE']:.6f} "
            f"| LLM={hybrid_metrics.get('linhas_llm')} "
            f"RN={hybrid_metrics.get('linhas_ranked_nodes')}",
            flush=True
        )

    # Evaluate the four individual conditions.
    for item,pred in predictions:
        record_result(item,pred)

    # Optional ensembles of the three personas, without additional queries.
    networks_by_repetition_method={}
    for item,pred in predictions:
        if item['condicao']!='controle':
            networks_by_repetition_method.setdefault(
                (item['repeticao'],item['metodo']),{}
            )[item['condicao']]=align_prediction_in_memory(pred,gold)

    original_persona_config=persona_config
    active_personas={
        p['id']:p
        for p in original_persona_config['personas']
        if p.get('ativo',True)
    }
    node_weights=original_persona_config.get(
        'calibracao',{}
    ).get('pesos_por_no',{})

    for (rep,method_name),network_models in networks_by_repetition_method.items():
        if len(network_models)<2:
            continue

        for pooling_mode in ('iguais','ponderados'):
            cpds=[]
            base_network=next(iter(network_models.values()))
            new_network=BayesianNetworkModel(list(base_network.edges()))
            new_network.add_nodes_from(base_network.nodes())
            effective_weights={}

            for node_name in base_network.nodes():
                base=base_network.get_cpds(node_name)

                if pooling_mode=='iguais':
                    ws={pid:1.0 for pid in network_models}
                else:
                    specified_weights=node_weights.get(node_name,{})
                    ws={
                        pid:float(specified_weights.get(pid,active_personas[pid]['peso']))
                        for pid in network_models
                    }

                total=sum(ws.values())
                if total<=0:
                    raise ValueError(
                        'Pesos da fusão são todos zero no nó '+node_name
                    )

                ws={pid:w/total for pid,w in ws.items()}
                effective_weights[node_name]=ws
                vals=np.zeros_like(
                    np.asarray(base.get_values(),dtype=float)
                )

                for pid,pred in network_models.items():
                    cp=pred.get_cpds(node_name)
                    if (
                        cp.variables!=base.variables
                        or cp.state_names!=base.state_names
                    ):
                        raise ValueError(
                            f'CPTs individuais incompatíveis na fusão {node_name}/{pid}'
                        )
                    vals+=ws[pid]*np.asarray(
                        cp.get_values(),dtype=float
                    )

                args=dict(
                    variable=node_name,
                    variable_card=base.variable_card,
                    values=vals,
                    state_names={
                        v:list(base.state_names[v])
                        for v in base.variables
                    },
                )
                if len(base.variables)>1:
                    args.update(
                        evidence=list(base.variables[1:]),
                        evidence_card=[
                            len(base.state_names[p])
                            for p in base.variables[1:]
                        ],
                    )

                cpds.append(TabularCPD(**args))

            new_network.add_cpds(*cpds)
            if not new_network.check_model():
                raise RuntimeError('Fusão inválida.')

            record_result(
                {
                    'condicao':'ensemble_'+pooling_mode,
                    'metodo':method_name,
                    'repeticao':rep,
                },
                new_network,
                derived=True,
                persona_weights=effective_weights,
            )

    # Complete report per generation: one record per node, checking every cell.
    audit_frame=pd.DataFrame(all_alignment_audits)
    audit_frame.to_csv(alignment_audit_dir/'auditoria_alinhamento_geracoes.csv',
                        index=False,encoding='utf-8-sig')
    audit_summary=(
        audit_frame.groupby(['condicao','repeticao'],as_index=False)
        .agg(nos_verificados=('no','nunique'),
             probabilidades_verificadas=('n_probabilidades_verificadas','sum'),
             nos_com_estados_filho_invertidos=('algum_estado_filho_invertido','sum'),
             nos_com_estados_pais_invertidos=('algum_estado_pai_invertido','sum'),
             nos_com_ordem_de_pais_alterada=('ordem_pais_alterada','sum'),
             erro_maximo=('erro_maximo_pos_alinhamento','max'))
    )
    audit_summary.to_csv(alignment_audit_dir/'auditoria_alinhamento_resumo.csv',
                            index=False,encoding='utf-8-sig')
    if len(audit_frame) != len(gold.nodes())*len(evaluation_outputs):
        raise AssertionError('Auditoria incompleta: número de nós/condições insuficiente.')
    if float(audit_frame['erro_maximo_pos_alinhamento'].max()) > 1e-10:
        raise AssertionError('Auditoria: houve divergência ao permutar índices.')
    print('[AUDITORIA] Todas as probabilidades verificadas por rótulos e configurações de pais.')
    display(audit_summary)

    evaluation_summary=pd.DataFrame(evaluation_outputs)

    # Reports identify conditions explicitly and separate generations from ensembles.
    column_order=[
        'modelo','rede','metodo','repeticao',
        'condicao','rotulo_condicao','grupo_condicao','persona_numero',
        'persona_id','persona_papel','persona_especialidade',
        'BN_KL_bits','CPT_KL_media_bits','CPT_KL_mediana_bits','CPT_KL_p90_bits',
        'MAE','RMSE','MSE','erro_maximo','vies_medio',
        'erro_absoluto_mediana','erro_absoluto_p90',
        'TVD_media','TVD_mediana','Hellinger_media','JS_media_bits','JS_p90_bits',
        'pct_probabilidades_erro_ate_005','pct_probabilidades_erro_ate_010',
        'n_linhas_cpt','n_probabilidades',
        'linhas_llm','linhas_ranked_nodes','linhas_fallback_uniforme',
        'percentual_llm','percentual_ranked_nodes',
        'percentual_fallback_uniforme','confianca_media','confianca_mediana'
    ]
    selected_columns=[col for col in column_order if col in evaluation_summary.columns]
    remaining_columns=[col for col in evaluation_summary.columns if col not in selected_columns]
    overall_comparison=evaluation_summary[selected_columns+remaining_columns].copy()
    _save_csv(DIR_AVALIACAO/'comparacao_geral.csv',overall_comparison)
    _save_csv(DIR_AVALIACAO/'comparacao_geracoes_individuais.csv',
        overall_comparison[overall_comparison['grupo_condicao'].isin(['Controle','Persona'])])
    _save_csv(DIR_AVALIACAO/'comparacao_ensembles.csv',
        overall_comparison[overall_comparison['grupo_condicao']=='Ensemble'])
    _save_csv(DIR_AVALIACAO/'indice_geracoes.csv',generation_index)
    _save_csv(DIR_AVALIACAO/'estatisticas_por_condicao.csv',
        _aggregate_repetitions(evaluation_summary))
    _save_csv(DIR_AVALIACAO/'ranking_por_metrica.csv',_rank_metrics(evaluation_summary))
    _save_text(DIR_AVALIACAO/'relatorio_geral.md',_overall_report(evaluation_summary,persona_map))
    _save_text(DIR_AVALIACAO/'dicionario_metricas.md',_metric_dictionary())

    evaluation_summary.to_csv(
        DIR_AVALIACAO/'comparacao_resumo.csv',
        index=False,
        encoding='utf-8-sig'
    )
    _save_json_atomic(
        DIR_AVALIACAO/'comparacao_resumo.json',
        evaluation_outputs
    )

    pd.DataFrame(all_row_metrics).to_csv(
        DIR_AVALIACAO/'metricas_por_linha.csv',
        index=False,
        encoding='utf-8-sig'
    )
    pd.DataFrame(all_node_metrics).to_csv(
        DIR_AVALIACAO/'metricas_por_no.csv',
        index=False,
        encoding='utf-8-sig'
    )
    pd.DataFrame(all_probability_errors).to_csv(
        DIR_AVALIACAO/'erros_por_probabilidade.csv',
        index=False,
        encoding='utf-8-sig'
    )
    pd.DataFrame(all_source_metrics).to_csv(
        DIR_AVALIACAO/'metricas_por_origem_llm_rn.csv',
        index=False,
        encoding='utf-8-sig'
    )
    pd.DataFrame(all_hybrid_metrics).to_csv(
        DIR_AVALIACAO/'resumo_hibrido.csv',
        index=False,
        encoding='utf-8-sig'
    )

    _save_json_atomic(
        DIR_AVALIACAO/'manifesto_avaliacao.json',
        {
            'criado_em_utc':datetime.now(timezone.utc).isoformat(),
            'geracao_manifesto':str(CAMINHO_MANIFESTO),
            'sha256_manifesto_geracao':hashlib.sha256(
                CAMINHO_MANIFESTO.read_bytes()
            ).hexdigest(),
            'schema_geracao':MANIFESTO.get('schema'),
            'modelo':MANIFESTO.get('modelo'),
            'metodo':MANIFESTO.get('configuracao',{}).get('metodo'),
            'rede':NOME_REDE_BNREP,
            'origem_referencia':reference_source,
            'bif_referencia':str(bif),
            'sha256_bif':hashlib.sha256(
                Path(bif).read_bytes()
            ).hexdigest(),
            'estado':'concluido',
            'permutacoes_estados_em_memoria':state_permutations,
            'condicoes_avaliadas':len(evaluation_outputs),
            'sem_dados_observados':True,
        }
    )

    display_columns=[
        'modelo','rotulo_condicao','repeticao',
        'BN_KL_bits','CPT_KL_media_bits','MAE','RMSE',
        'JS_media_bits','Hellinger_media','TVD_media',
        'linhas_llm','linhas_ranked_nodes','linhas_fallback_uniforme'
    ]
    display(evaluation_summary[[c for c in display_columns if c in evaluation_summary.columns]])
    print('\n[PERSONAS IDENTIFICADAS]')
    for pid,ident in persona_map.items():
        print(f"  Persona {ident['persona_numero']}: {ident['persona_papel']} ({pid})")

    print('[AVALIAÇÃO CONCLUÍDA]',DIR_AVALIACAO,flush=True)
    print('[ARQUIVOS PRINCIPAIS]')
    for item_name in [
        'comparacao_geral.csv',
        'comparacao_geracoes_individuais.csv',
        'comparacao_ensembles.csv',
        'estatisticas_por_condicao.csv',
        'ranking_por_metrica.csv',
        'indice_geracoes.csv',
        'mapa_personas.csv',
        'relatorio_geral.md',
        'comparacao_resumo.csv',
        'resumo_hibrido.csv',
        'metricas_por_origem_llm_rn.csv',
        'metricas_por_no.csv',
        'metricas_por_linha.csv',
        'erros_por_probabilidade.csv',
    ]:
        print('  -',DIR_AVALIACAO/item_name)

    return evaluation_summary


In [ ]:
RESUMO_AVALIACAO=evaluate_and_save()

[REDE VALIDADA] electricvehicle: 23 nós, 22 arcos, 79 linhas CPT por geração.
[PREFLIGHT] CPTs estimadas válidas: 4 — agora podemos consultar bnRep.
[bnRep] Exportando rede oficial por R:  electricvehicle
Na primeira execução a instalação pode levar muitos minutos.
[bnRep] Instalando/carregando pacote e exportando referência. Acompanhe a saída:
[R] Installing package into ‘/content/R_libs’
[R] (as ‘lib’ is unspecified)
[R] trying URL 'https://cloud.r-project.org/src/contrib/BiocManager_1.30.27.tar.gz'
[R] Content type 'application/x-gzip' length 752490 bytes (734 KB)
[R] ==================================================
[R] downloaded 734 KB
[R] 
[R] * installing *source* package ‘BiocManager’ ...
[R] ** this is package ‘BiocManager’ version ‘1.30.27’
[R] ** package ‘BiocManager’ successfully unpacked and MD5 sums checked
[R] ** using staged installation
[R] ** R
[R] *** installing help indices
[R] ** installing vignettes
[R] ** testing if installed package can be loaded from temporar

,condicao,repeticao,nos_verificados,probabilidades_verificadas,nos_com_estados_filho_invertidos,nos_com_estados_pais_invertidos,nos_com_ordem_de_pais_alterada,erro_maximo
0,analista_risco,1,23,158,23,8,6,0.0
1,controle,1,23,158,23,8,6,0.0
2,engenheiro_veiculos_eletricos,1,23,158,23,8,6,0.0
3,ensemble_iguais,1,23,158,0,0,0,0.0
4,ensemble_ponderados,1,23,158,0,0,0,0.0
5,especialista_seguranca_automotiva,1,23,158,23,8,6,0.0


,modelo,rotulo_condicao,repeticao,BN_KL_bits,CPT_KL_media_bits,MAE,RMSE,JS_media_bits,Hellinger_media,TVD_media,linhas_llm,linhas_ranked_nodes,linhas_fallback_uniforme
0,Qwen/Qwen3.5-9B,Controle — sem persona,1,10.643427,1.644063,0.417034,0.489239,0.223046,0.364974,0.417034,19.0,60.0,0.0
1,Qwen/Qwen3.5-9B,Persona 1 — Electric vehicle systems engineer,1,22.697218,2.115589,0.399229,0.510505,0.243252,0.359780,0.399229,38.0,41.0,0.0
2,Qwen/Qwen3.5-9B,Persona 2 — Automotive fire safety and reliabi...,1,13.139460,1.548242,0.397052,0.489669,0.227745,0.363802,0.397052,33.0,46.0,0.0
3,Qwen/Qwen3.5-9B,Persona 3 — Probabilistic risk analyst,1,24.042911,2.954032,0.427730,0.515394,0.256184,0.395497,0.427730,32.0,47.0,0.0
4,Qwen/Qwen3.5-9B,Ensemble — pesos iguais,1,11.844375,0.943463,0.398551,0.479786,0.217920,0.358625,0.398551,NaN,NaN,NaN
5,Qwen/Qwen3.5-9B,Ensemble — pesos das personas,1,11.564656,0.936982,0.396750,0.481455,0.217556,0.355888,0.396750,NaN,NaN,NaN



[PERSONAS IDENTIFICADAS]
  Persona 1: Electric vehicle systems engineer (engenheiro_veiculos_eletricos)
  Persona 2: Automotive fire safety and reliability specialist (especialista_seguranca_automotiva)
  Persona 3: Probabilistic risk analyst (analista_risco)
[AVALIAÇÃO CONCLUÍDA] /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep
[ARQUIVOS PRINCIPAIS]
  - /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep/comparacao_geral.csv
  - /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep/comparacao_geracoes_individuais.csv
  - /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep/comparacao_ensembles.csv
  - /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep/estatisticas_por_condicao.csv
  - /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep/ranking_por_metrica.csv
  - /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep/indice_geracoes.csv
  - /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep/mapa_pe

## 7. Final results — quick guide

1. **`relatorio_geral.md`**: a short explanation and comparison table using the actual persona names.
2. **`comparacao_geral.csv`**: all overall metrics, with one row per generation/ensemble and repetition.
3. **`comparacao_geracoes_individuais.csv`**: control + persona 1 + persona 2 + persona 3, excluding ensembles.
4. **`comparacao_ensembles.csv`**: equal-weight and persona-weighted ensembles only.
5. **`por_geracao/`**: a folder for each condition, containing its summary, metrics by node, row, and source, and errors per probability.
6. **`estatisticas_por_condicao.csv`**: means, standard deviations, minima/maxima, and medians when more than one repetition is available.
7. **`dicionario_metricas.md`**: mathematical/conceptual definitions of the fields.

Previous CSV outputs remain available. The last cell packages everything into a ZIP. With a single repetition, the standard deviation is undefined (blank), not zero.


In [ ]:
print("ARQUIVOS DE AVALIAÇÃO:")
for output_file in sorted(DIR_AVALIACAO.rglob("*.json")):
    print(" •",output_file.relative_to(DIR_AVALIACAO))

ARQUIVOS DE AVALIAÇÃO:
 • comparacao_resumo.json
 • manifesto_avaliacao.json
 • referencia_bnrep/electricvehicle_manifesto.json
 • relatorios/metricas_analista_risco_ConfidenceRankedNodes_r1.json
 • relatorios/metricas_controle_ConfidenceRankedNodes_r1.json
 • relatorios/metricas_engenheiro_veiculos_eletricos_ConfidenceRankedNodes_r1.json
 • relatorios/metricas_ensemble_iguais_ConfidenceRankedNodes_r1.json
 • relatorios/metricas_ensemble_ponderados_ConfidenceRankedNodes_r1.json
 • relatorios/metricas_especialista_seguranca_automotiva_ConfidenceRankedNodes_r1.json


## 7.1. Check the state audit before publishing metrics

The report shows the number of nodes and probabilities checked per generation. `erro_maximo` must be zero or within the `1e-10` precision tolerance. The number of state-order reversals **is not an error**: it indicates that the notebook detected a different order and handled it by matching names.


In [ ]:
# @title Audit — states, parent order, and equivalence of every probability
AUDITORIA_DIR = DIR_AVALIACAO/'auditoria_alinhamento'
for item_name in ('auditoria_estados_bnrep.csv', 'auditoria_alinhamento_resumo.csv'):
    output_file = AUDITORIA_DIR/item_name
    print('\n[RELATÓRIO]', output_file)
    display(pd.read_csv(output_file))
print('\n[DETALHE POR NÓ] Primeiras 15 linhas; arquivo completo salvo em CSV.')
display(pd.read_csv(AUDITORIA_DIR/'auditoria_alinhamento_geracoes.csv').head(15))



[RELATÓRIO] /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep/auditoria_alinhamento/auditoria_estados_bnrep.csv


,no,ordem_json,ordem_bnrep,indices_json_na_ordem_bnrep,permutacao_necessaria,estados_compativeis
0,ACF,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",True,True
1,AM,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",True,True
2,AWE,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",True,True
3,BEP,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",True,True
4,BF,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",True,True
5,BO,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",True,True
6,CBI,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",True,True
7,CEF,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",True,True
8,CI,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",True,True
9,DTH,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",True,True



[RELATÓRIO] /content/entrada_qwen35_confidence_rn/extraido/avaliacao_bnrep/auditoria_alinhamento/auditoria_alinhamento_resumo.csv


,condicao,repeticao,nos_verificados,probabilidades_verificadas,nos_com_estados_filho_invertidos,nos_com_estados_pais_invertidos,nos_com_ordem_de_pais_alterada,erro_maximo
0,analista_risco,1,23,158,23,8,6,0.0
1,controle,1,23,158,23,8,6,0.0
2,engenheiro_veiculos_eletricos,1,23,158,23,8,6,0.0
3,ensemble_iguais,1,23,158,0,0,0,0.0
4,ensemble_ponderados,1,23,158,0,0,0,0.0
5,especialista_seguranca_automotiva,1,23,158,23,8,6,0.0



[DETALHE POR NÓ] Primeiras 15 linhas; arquivo completo salvo em CSV.


,condicao,repeticao,no,ordem_estados_predicao,ordem_estados_bnrep,permutacao_indices_filho,ordem_pais_predicao,ordem_pais_bnrep,algum_estado_filho_invertido,algum_estado_pai_invertido,ordem_pais_alterada,n_probabilidades_verificadas,erro_maximo_pos_alinhamento,resultado
0,controle,1,ACF,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",[],[],True,False,False,2,0.0,OK — equivalência por rótulo verificada
1,controle,1,AM,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",[],[],True,False,False,2,0.0,OK — equivalência por rótulo verificada
2,controle,1,AWE,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",[],[],True,False,False,2,0.0,OK — equivalência por rótulo verificada
3,controle,1,BEP,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",[],[],True,False,False,2,0.0,OK — equivalência por rótulo verificada
4,controle,1,BF,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",[],[],True,False,False,2,0.0,OK — equivalência por rótulo verificada
5,controle,1,BO,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",[],[],True,False,False,2,0.0,OK — equivalência por rótulo verificada
6,controle,1,CBI,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]","[""IS"", ""CI""]","[""CI"", ""IS""]",True,True,True,8,0.0,OK — equivalência por rótulo verificada
7,controle,1,CEF,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",[],[],True,False,False,2,0.0,OK — equivalência por rótulo verificada
8,controle,1,CI,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",[],[],True,False,False,2,0.0,OK — equivalência por rótulo verificada
9,controle,1,DTH,"[""no"", ""yes""]","[""yes"", ""no""]","[1, 0]",[],[],True,False,False,2,0.0,OK — equivalência por rótulo verificada


## 8. Download the complete evaluation

The next cell packages the global reports and every `por_geracao/` and `tabelas_cpt_completas/` folder into a ZIP. It does not require Qwen, a GPU, or Google Drive.


In [ ]:
# @title 8.1 — Package and download the evaluation
import shutil
from pathlib import Path

archive_base_path = Path("/content") / f"avaliacao_bnrep_{NOME_REDE_BNREP}"
zip_path = shutil.make_archive(
    str(archive_base_path),
    "zip",
    root_dir=str(DIR_AVALIACAO)
)

print("[ZIP]", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Fora do Colab; arquivo disponível em:", zip_path)


[ZIP] /content/avaliacao_bnrep_electricvehicle.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>